<div align="center">

**Türkçe:** Bu eğitim Türkçe yürütülmektedir. [Bu Jupyter Notebook'un Türkçesini Colab'da açın](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-01/NB01_python_ve_ekg_temelleri.ipynb) · [Türkçe ders sayfası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html)

# Week 01: Python and cardiology data basics

**Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists**<br>
Turkish Society of Cardiology · Prof. Dr. Utku Kose

[Week page](README.md) · [Lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html) · [Interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lab.html) · [PDF notes](Week01_Lecture_Notes.pdf) · [Open in Colab](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-01/NB01_python_and_ecg_basics.ipynb)

</div>

**This notebook has two parts.** In the first, an artificial intelligence (AI) assistant writes the code from a prompt, a written instruction. In the second, you write the code yourself in Python. Both parts reach the same results with the same data, so choose one of them at each practice stop of the lecture page.

**Together with the lecture page.** You come to this Jupyter notebook from the practice stops of the lecture page. At the end of every stop, a link takes you back to the lecture page, to the place where reading continues.

| | For whom? | What you do |
|---|---|---|
| **[Part 1: Prompt path](#part-1)** | Participants who do not write code | An AI assistant writes the code from ready prompts; you run it and check the result. |
| **[Part 2: Python path](#part-2)** | Participants who want to learn Python | You write and run the code step by step, with exercises. |

**Where it runs.** The same notebook runs in Google Colab, with nothing to install, and in Jupyter on your own computer (Anaconda; see SETUP.md in the course folder). Choose the one you prefer. In Colab, save your own copy first with *File → Save a copy in Drive*; otherwise the changes are lost when the tab is closed. Run Section 0 first, whichever part you choose. It prepares the course files and the packages.

**Data and privacy.** Every data set here is synthetic or comes from a public research database. Real patient data belongs only on your own computer, under the rules of your institution and with the approval of an ethics committee, and never in a cloud notebook or in an AI chat.

## 0. Setup

**First use of Jupyter Notebook.** A notebook is made of cells. This cell holds text; the grey cells below hold Python code. Click into a code cell and press Shift+Enter: The cell runs, its result appears below it, and the number in the brackets on its left, such as `[1]`, shows the order of the run. The kernel, the Python process behind the notebook, remembers every name that a cell has defined, so the cells run from top to bottom. **Kernel → Restart Kernel and Run All Cells** repeats the whole notebook, and Ctrl+S (Cmd+S on a Mac) saves it as an `.ipynb` file.

The first cell finds the course folder. If the notebook was opened on its own, for example in Google Colab, it downloads the course package `vclinic` and the data files from GitHub once. The second cell loads the packages and chooses the language of the messages. It also checks the two ECG packages of the week, `wfdb` and `neurokit2`, and installs them once with pip when they are missing.

In [ ]:
#@title Course files: run this cell (there is no need to open it)
import pathlib
import sys
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/"
DATA_FILES = [
    "data/uci-heart-disease/processed.cleveland.data",
    "data/physionet/mitdb/100.hea",
    "data/physionet/mitdb/100.dat",
    "data/physionet/mitdb/100.atr",
    "data/physionet/ptbdb/patient001/s0010_re.hea",
    "data/physionet/ptbdb/patient001/s0010_re.dat",
    "data/physionet/ptbdb/patient001/s0010_re.xyz",
    "data/examples/example_caliper_measurements.csv",
]


def find_course_root():
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "vclinic" / "__init__.py").exists():
            return folder
    return None


ROOT = find_course_root()
if ROOT is None or ROOT.name == "ai-cardiology-course":
    # A downloaded copy, for example in Colab: the files of the package are refreshed.
    ROOT = pathlib.Path.cwd() / "ai-cardiology-course" if ROOT is None else ROOT
    manifest = urllib.request.urlopen(REPO_RAW + "vclinic/MANIFEST.txt").read().decode("utf-8")
    package = ["vclinic/" + line.strip() for line in manifest.splitlines() if line.strip()]
    for rel in package + DATA_FILES:
        target = ROOT / rel
        if rel in package or not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(REPO_RAW + rel, target)
    print("Course files downloaded to", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# Setup: packages, language and plotting style
import importlib
import importlib.util

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.set_language("en")
plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans"})
RANDOM_STATE = 2026

print("vclinic version", vclinic.__version__)
print("Course folder:", ROOT)
import subprocess
import sys

for name in ["wfdb", "neurokit2", "ipywidgets"]:
    try:
        importlib.import_module(name)
        print(f"{name}: installed")
    except ImportError:
        if name in ("wfdb", "neurokit2"):
            # the two ECG packages of the week are installed once with pip
            result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", name], capture_output=True, text=True)
            importlib.invalidate_caches()
            ok = result.returncode == 0 and importlib.util.find_spec(name) is not None
            print(f"{name}: " + ("installed now" if ok else "could not be installed (no internet?); the notebook uses a built-in fallback"))
        else:
            print(f"{name}: not installed (optional)")

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
teach.register_answers({
    "Exercise 1.1": (0.833, 0.001),
    "Exercise 1.2": (439.1, 0.2),
    "Exercise 1.3": (2, 0),
    "Exercise A": (56.6, 0.1),
    "Exercise B": (32.0, 0.1),
    "Exercise C": (47, 0),
    "Exercise D": (100, 0),
    "Exercise E": (0.104, 0.002),
    "Exercise F": (403.9, 0.3),
    "Exercise G": (216, 6),
    "Exercise H": (33, 0),
    "Exercise I": (39.05, 0.1),
    "Exercise J": (466.5, 0.2),
    "Step 1.1": (56.6, 0.05),
    "Step 1.2": (32.0, 0.05),
    "Step 1.3": (47, 0),
    "Step 2.1": (1250, 1),
    "Step 2.2": (84.9, 0.5),
    "Step 2.3": (403.9, 0.05),
    "Step 3.1": (33, 0),
    "Step 3.2": (36.0, 0.1),
    "Step 3.3": (81.8, 0.5),
    "Step 3.4": (466.5, 0.05),
})
print("Expected answers are ready.")

<a id="part-1"></a>

# Part 1: Prompt path

**For whom?** This part is for participants who do not write code. Here you type no code. An artificial intelligence (AI) assistant writes it from a ready prompt, a written instruction; you run the code and check the result.

**What you need.** An AI assistant that can write Python code is enough. Gemini is built into Google Colab: Open it with the Gemini button at the top of the window. ChatGPT, Claude, Copilot or Gemini in another browser tab work as well.

**How a step works.**

1. Read the goal of the step.
2. Run the data cell under the prompt. It prepares the table "data" that the prompt describes and shows its first and last rows.
3. Copy the prompt: Select it with the mouse and press Ctrl+C (Cmd+C on a Mac).
4. Paste it into the AI assistant and send it.
5. Copy the code that the assistant returns, paste it into the cell under the data cell in place of the line that starts with `#`, and run the cell with the ▶ button or with Shift+Enter.
6. Run the check cell below it. It prints *correct ✓* or *not yet ✗*.
7. If it prints *not yet*, send the repair prompt below to the assistant together with the code. If the result is still wrong, run the *Ready code* cell. It always gives the correct result, so the next step can start.

**Two names in every prompt: "data" and "answer".** An AI assistant usually does not see the notebook, so every prompt describes the data in words. In each step, the data cell puts the data of the step into a table named "data", and the prompt describes the rows and the columns of this table. The prompt also asks the assistant to keep the result under the name "answer", and the check cell reads the result under this name. These are the only two names from the code that appear in the prompts of the course.

**At the start of every stop** run its preparation cell once. It loads the data of the stop, so a stop can also be started on another day.

**Repair prompt** (for code that runs but does not pass the check):

> The code below was written for this task: [paste the Task line of the prompt]. It runs, but the check of the course prints "not yet". Find the error, explain it in two sentences and return the corrected code. The result must still be kept under the name "answer".  
>
> [paste the code here]  

**Prompt for an error message** (for code that stops with an error):

> Running the code below in a Jupyter notebook gives the error message that follows. Explain the cause in plain language and return the corrected code.  
>
> [paste the code here]  
>
> [paste the error message here]  

All prompts of the week are also in the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-01/PROMPTS.md), where GitHub shows a copy button next to each one.

**A rule that never changes.** The prompts contain only the synthetic or public data of the course. Never paste real patient data into an AI assistant.

## Stop 1: Mini project: The Kaggle Heart Disease data set

*Lecture: [Practice stop 1](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#stop-1)* · *The same result on the Python path: Part 2, Sections 1 to 2*

This stop is the mini project of the session: loading the Kaggle Heart Disease data set, the Cleveland data of the UCI Heart Disease collection, with a statistical summary and a visualisation. The registry of a clinic is a table: one row per patient, one column per variable. In this stop, the 303 patients of the data set and the first 500 patients of the virtual clinic are summarised by a mean, a share and a count, each with a clinical question behind it.

**At the end of this stop:** The mean age of the Cleveland patients with coronary artery disease (56.6 years), the share of women among all 303 patients (32.0 percent) and the number of patients with HFrEF in the virtual clinic (47), with a box plot of the ejection fraction by diagnosis.

In [ ]:
#@title Preparation of Stop 1: run this cell once (it loads the data of the stop)
from vclinic.datasets import load_cleveland
heart = load_cleveland()
clinic = vclinic.make_cohort(n=500, seed=RANDOM_STATE)
print("Stop 1 is ready.")

### Step 1.1: The mean age of the patients with disease

**Goal.** Find out how old, on average, the Cleveland patients with coronary artery disease were.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know the mean age of the patients with coronary artery disease among 303 patients who underwent coronary angiography at the Cleveland Clinic.  
> Data: The notebook holds a table named "data". Each row is a patient: The column "age" gives the age in years. The column "CAD" says "yes" when at least one major coronary artery was narrowed by more than 50 percent of its diameter at angiography, and "no" otherwise.  
> Task: Select the patients with coronary artery disease and compute their mean age. Also print how many patients were selected.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the mean age in years with one decimal and keep it under the name "answer".  
> Test: For comparison, also print the mean age of all patients, which must be 54.4 years. The mean of the patients with disease must differ from it.  

In [ ]:
#@title Data of step 1.1: run this cell (it prepares the table "data" and shows it)
data = pd.DataFrame({"age": heart["age"],
                     "CAD": heart["disease"].map({1: "yes", 0: "no"})})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.1
teach.check_step("Step 1.1", "answer", globals())

In [ ]:
#@title Ready code of step 1.1 (run it only if the code of the assistant does not pass the check)
data = pd.DataFrame({"age": heart["age"],
                     "CAD": heart["disease"].map({1: "yes", 0: "no"})})
patients = data[data["CAD"] == "yes"]
print("Patients with disease:", len(patients))
print("Mean age of all patients:", round(data["age"].mean(), 1))
answer = round(patients["age"].mean(), 1)
print("Mean age of the patients with disease:", answer)
teach.check_step("Step 1.1", "answer", globals())

**What the result shows.** The patients with disease are about two years older than the whole group. A single mean says little about a single patient; the tables of the lecture page show the spread as well.

### Step 1.2: The share of women

**Goal.** Find out which percentage of the Cleveland patients were women.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know which percentage of the 303 patients who underwent coronary angiography at the Cleveland Clinic were women.  
> Data: The notebook holds a table named "data". Each row is a patient: The column "age" gives the age in years, and the column "sex" says "female" or "male".  
> Task: Divide the number of women by the number of all patients and express the share in percent. Compute the percentage of men in the same way.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the percentage of women with one decimal and keep it under the name "answer". Also print the percentage of men.  
> Test: The two percentages must add up to 100.  

In [ ]:
#@title Data of step 1.2: run this cell (it prepares the table "data" and shows it)
data = pd.DataFrame({"age": heart["age"],
                     "sex": heart["sex"].map({1: "male", 0: "female"})})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.2
teach.check_step("Step 1.2", "answer", globals())

In [ ]:
#@title Ready code of step 1.2 (run it only if the code of the assistant does not pass the check)
data = pd.DataFrame({"age": heart["age"],
                     "sex": heart["sex"].map({1: "male", 0: "female"})})
women = 100 * (data["sex"] == "female").mean()
men = 100 * (data["sex"] == "male").mean()
answer = round(women, 1)
print("Women (%):", answer, " Men (%):", round(men, 1))
teach.check_step("Step 1.2", "answer", globals())

**What the result shows.** About two thirds of the Cleveland patients are men. A model trained on such data learns less about women, and Week 9 treats this as a question of bias.

### Step 1.3: Patients with HFrEF in the virtual clinic

**Goal.** Count the patients of the virtual clinic who have heart failure with reduced ejection fraction, and see the ejection fraction of every diagnosis in one figure.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to count the patients with heart failure with reduced ejection fraction (HFrEF) among the 500 synthetic patients of the virtual clinic, and to see how the ejection fraction differs between the diagnoses.  
> Data: The notebook holds a table named "data". Each row is a patient: The column "patient" gives the number of the patient, the column "main diagnosis" the main diagnosis and the column "LVEF" the left ventricular ejection fraction in percent. The main diagnosis is one of "none", "CCS" (chronic coronary syndrome), "ACS" (acute coronary syndrome), "HFrEF" (heart failure with reduced ejection fraction) and "HFpEF" (heart failure with preserved ejection fraction).  
> Task: Count the patients whose main diagnosis is HFrEF. Then draw a box plot of the ejection fraction for each main diagnosis, with the diagnoses on the horizontal axis and the ejection fraction in percent on the vertical axis.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the number of patients with HFrEF and keep it under the name "answer". Show the plot as well.  
> Test: Also print the number of patients of every diagnosis; the five numbers must add up to 500. No patient with HFrEF may have an ejection fraction above 40 percent.  

In [ ]:
#@title Data of step 1.3: run this cell (it prepares the table "data" and shows it)
data = pd.DataFrame({"patient": clinic["patient_id"],
                     "main diagnosis": clinic["main_diagnosis"].replace({"none": "none"}),
                     "LVEF": clinic["lvef"]})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.3
teach.check_step("Step 1.3", "answer", globals())

In [ ]:
#@title Ready code of step 1.3 (run it only if the code of the assistant does not pass the check)
data = pd.DataFrame({"patient": clinic["patient_id"],
                     "main diagnosis": clinic["main_diagnosis"].replace({"none": "none"}),
                     "LVEF": clinic["lvef"]})
print(data["main diagnosis"].value_counts())
answer = int((data["main diagnosis"] == "HFrEF").sum())
order = ["none", "CCS", "ACS", "HFpEF", "HFrEF"]
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.boxplot([data.loc[data["main diagnosis"] == d, "LVEF"] for d in order])
ax.set_xticks(range(1, 6), order)
ax.set_xlabel("Main diagnosis")
ax.set_ylabel("LVEF (%)")
plt.show()
print("Patients with HFrEF:", answer)
teach.check_step("Step 1.3", "answer", globals())

**What the result shows.** In the registry, every patient with HFrEF has an ejection fraction of 40 percent or less, as the definition of the European Society of Cardiology requires, and every patient with HFpEF one of 50 percent or more. Heart failure with mildly reduced ejection fraction (41 to 49 percent) is left out to keep the virtual clinic simple.

**Stop 1 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#last-two-hours-working-with-ecg-data) and continue reading at «Last two hours: Working with ECG data». At the next practice stop, the lecture page brings you back to this Jupyter notebook.

## Stop 2: The ECG as data

*Lecture: [Practice stop 2](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#stop-2)* · *The same result on the Python path: Part 2, Sections 3 to 5*

A digital ECG is a table of numbers: one column per lead, one row per sample. In this stop, lead II of the ECG of the case of the week, VC-0317, is drawn and measured, and the QT interval of another patient is corrected for the heart rate by two formulas.

**At the end of this stop:** A strip of lead II with 1250 samples in 2.5 seconds, a heart rate of 84.9 beats per minute from the R peaks, and a QTc of 403.9 ms by Fridericia's formula.

In [ ]:
#@title Preparation of Stop 2: run this cell once (it loads the data of the stop)
clinic = vclinic.make_cohort(n=500, seed=RANDOM_STATE)
case = vclinic.patient_ecg(clinic, "VC-0317")
print("Stop 2 is ready.")

### Step 2.1: A strip of lead II

**Goal.** Draw the first 2.5 seconds of lead II and see how many numbers such a strip holds.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to see the first 2.5 seconds of lead II from the 10-second resting ECG of a 67-year-old woman, and to know how many samples this strip holds.  
> Data: The notebook holds a table named "data". Each row is one moment of the recording: The column "time" gives the time in seconds, and the column "II" the voltage of lead II in millivolts. The recording was sampled 500 times per second.  
> Task: Keep the rows whose time is less than 2.5 seconds and count them. Draw this part with the time in seconds on the horizontal axis and the voltage in millivolts on the vertical axis.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the number of samples in these 2.5 seconds and keep it under the name "answer". Show the plot as well.  
> Test: The time axis of the plot must start at 0 and end at 2.5 seconds.  

In [ ]:
#@title Data of step 2.1: run this cell (it prepares the table "data" and shows it)
data = pd.DataFrame({"time": case.time, "II": case.lead("II")})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 2.1
teach.check_step("Step 2.1", "answer", globals())

In [ ]:
#@title Ready code of step 2.1 (run it only if the code of the assistant does not pass the check)
data = pd.DataFrame({"time": case.time, "II": case.lead("II")})
strip = data[data["time"] < 2.5]
answer = len(strip)
fig, ax = plt.subplots(figsize=(8, 2.6))
ax.plot(strip["time"], strip["II"], color="k", lw=0.9)
ax.set_xlim(0, 2.5)
ax.set_xlabel("Time (s)")
ax.set_ylabel("mV")
plt.show()
print("Samples in 2.5 s:", answer)
teach.check_step("Step 2.1", "answer", globals())

**What the result shows.** At 500 samples per second, 2.5 seconds are 1250 numbers for each lead, and the whole 10-second recording of 12 leads holds 60,000 numbers. The strip on the screen is a picture of this table.

### Step 2.2: The heart rate from the R peaks

**Goal.** Find the R peaks of lead II and compute the heart rate from the intervals between them.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to measure the heart rate of a 67-year-old woman from her 10-second resting ECG.  
> Data: The notebook holds a table named "data". Each row is one moment of the recording: The column "time" gives the time in seconds, and the column "II" the voltage of lead II in millivolts. The recording was sampled 500 times per second.  
> Task: Find the R peaks. Accept as R peaks the peaks that exceed half of the highest value and lie at least 0.3 seconds apart. Compute the heart rate in beats per minute from the mean time between successive R peaks.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the heart rate with one decimal and keep it under the name "answer". Draw lead II with the R peaks marked.  
> Test: About 14 R peaks must be found in the 10 seconds, and the heart rate must lie between 40 and 150 beats per minute.  

In [ ]:
#@title Data of step 2.2: run this cell (it prepares the table "data" and shows it)
data = pd.DataFrame({"time": case.time, "II": case.lead("II")})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 2.2
teach.check_step("Step 2.2", "answer", globals())

In [ ]:
#@title Ready code of step 2.2 (run it only if the code of the assistant does not pass the check)
data = pd.DataFrame({"time": case.time, "II": case.lead("II")})
from scipy.signal import find_peaks
t = data["time"].to_numpy()
x = data["II"].to_numpy()
r, _ = find_peaks(x, height=x.max() / 2, distance=int(0.3 * 500))
answer = round(60 / np.diff(t[r]).mean(), 1)
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.plot(t, x, color="k", lw=0.7)
ax.plot(t[r], x[r], "o", color="#B4312A")
ax.set_xlabel("Time (s)")
ax.set_ylabel("mV")
plt.show()
print("R peaks:", len(r), " heart rate (bpm):", answer)
teach.check_step("Step 2.2", "answer", globals())

**What the result shows.** The heart rate of the case is close to its value in the registry, 85 beats per minute. The rule of the prompt is simple: In a patient with tall T waves, it can count T waves as R peaks and double the heart rate. The lecture page shows this with the ECGs of other patients.

### Step 2.3: The QTc by two formulas

**Goal.** Correct the QT interval of patient VC-0254 for his fast heart rate with the formulas of Fridericia and Bazett, and compare the two.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to correct the QT interval of a 58-year-old man with a fast heart rate by two formulas and to compare the results.  
> Data: The notebook holds a table named "data" with one row: The column "patient" gives the number of the patient, the column "heart rate" the heart rate in beats per minute and the column "QT" the QT interval in milliseconds.  
> Task: Compute the RR interval in seconds as 60 divided by the heart rate. Then correct the QT interval with two formulas: Fridericia's formula divides the QT by the cube root of RR, and Bazett's formula divides it by the square root of RR.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the QTc of Fridericia in milliseconds with one decimal and keep it under the name "answer". Also print the QTc of Bazett.  
> Test: At a heart rate of 60 beats per minute, both formulas must give the QT itself.  

In [ ]:
#@title Data of step 2.3: run this cell (it prepares the table "data" and shows it)
data = pd.DataFrame({"patient": ["VC-0254"], "heart rate": [98], "QT": [343]})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 2.3
teach.check_step("Step 2.3", "answer", globals())

In [ ]:
#@title Ready code of step 2.3 (run it only if the code of the assistant does not pass the check)
data = pd.DataFrame({"patient": ["VC-0254"], "heart rate": [98], "QT": [343]})
rate = data["heart rate"].iloc[0]
qt = data["QT"].iloc[0]
rr = 60 / rate
answer = round(qt / rr ** (1 / 3), 1)
bazett = round(qt / rr ** 0.5, 1)
print("RR (s):", round(rr, 3), " QTc Fridericia:", answer, " QTc Bazett:", bazett)
teach.check_step("Step 2.3", "answer", globals())

**What the result shows.** At 98 beats per minute, Bazett's formula gives about 35 ms more than Fridericia's. Bazett's formula overcorrects at fast heart rates, which is why most studies of drug safety report Fridericia's correction.

**Stop 2 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#real-data-physionet-records) and continue reading at «Access to the PhysioNet and MIT-BIH data sets». At the next practice stop, the lecture page brings you back to this Jupyter notebook.

## Stop 3: Real recordings and a careless prompt

*Lecture: [Practice stop 3](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#stop-3)* · *The same result on the Python path: Part 2, Sections 6 to 9*

Real recordings bring their own noise, annotations and units. In this stop, record 100 of the MIT-BIH Arrhythmia Database is read with the beat labels of its cardiologists, and the heart rate variability is computed from its normal beats. Then a real 12-lead ECG of the PTB database is drawn. Finally, a QTc calculation that an AI assistant wrote from a short and careless prompt is requested again with the prompt card, and its result is checked.

**At the end of this stop:** The number of atrial premature beats in record 100 (33), the SDNN of its normal-to-normal intervals (36.0 ms), the heart rate of the PTB record (81.8 beats per minute) and a corrected QTc of 466.5 ms.

In [ ]:
#@title Preparation of Stop 3: run this cell once (it loads the data of the stop)
from vclinic.datasets import read_physionet, read_physionet_annotation
mit, fs_mit, notes = read_physionet("100", "mitdb")
ann_samples, ann_symbols = read_physionet_annotation("100", "mitdb")
ptb, fs_ptb, header = read_physionet("s0010_re", "ptbdb/patient001")
print("Stop 3 is ready.")

### Step 3.1: The beats of record 100

**Goal.** Count the beats of a 30-minute recording by the labels that cardiologists gave them.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: Record 100 of the MIT-BIH Arrhythmia Database is a 30-minute ambulatory ECG of a 69-year-old man. The cardiologist wants to count the beats of this recording by the labels that cardiologists gave them.  
> Data: The notebook holds a table named "data". Each row is a mark that the cardiologists placed on the recording: The column "time" gives its place in seconds, and the column "label" its kind. "N" marks a normal beat, "A" an atrial premature beat and "V" a premature ventricular contraction; "+" marks a change of rhythm and is not a beat.  
> Task: Count how often each label occurs and print the counts.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the number of atrial premature beats and keep it under the name "answer".  
> Test: The labels N, A and V together must count 2273 beats.  

In [ ]:
#@title Data of step 3.1: run this cell (it prepares the table "data" and shows it)
data = pd.DataFrame({"time": np.asarray(ann_samples) / fs_mit, "label": ann_symbols})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 3.1
teach.check_step("Step 3.1", "answer", globals())

In [ ]:
#@title Ready code of step 3.1 (run it only if the code of the assistant does not pass the check)
data = pd.DataFrame({"time": np.asarray(ann_samples) / fs_mit, "label": ann_symbols})
counts = data["label"].value_counts()
print(counts)
answer = int(counts.get("A", 0))
print("Atrial premature beats:", answer)
teach.check_step("Step 3.1", "answer", globals())

**What the result shows.** Record 100 is a quiet recording of a 69-year-old man: most beats are normal, with a few dozen atrial premature beats and a single premature ventricular contraction. The labels turn it into a reference against which a detector can be measured.

### Step 3.2: Heart rate variability: SDNN

**Goal.** Measure how much the intervals between normal beats vary over 30 minutes.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to measure the heart rate variability of record 100 of the MIT-BIH Arrhythmia Database, a 30-minute ambulatory ECG of a 69-year-old man, by the SDNN.  
> Data: The notebook holds a table named "data". Each row is a mark that the cardiologists placed on the recording: The column "time" gives its place in seconds, and the column "label" its kind. "N" marks a normal beat, "A" an atrial premature beat and "V" a premature ventricular contraction; "+" marks a change of rhythm and is not a beat.  
> Task: Keep only the beats, that is, all labels except "+". Compute the RR intervals between successive beats in milliseconds. Keep only the intervals between two normal beats, both labelled "N": These are the NN intervals. The SDNN is their standard deviation; compute it as for a sample, with the number of intervals minus one in the denominator. Also print the mean NN interval.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the SDNN in milliseconds with one decimal and keep it under the name "answer".  
> Test: The mean NN interval must lie between 700 and 900 ms.  

In [ ]:
#@title Data of step 3.2: run this cell (it prepares the table "data" and shows it)
data = pd.DataFrame({"time": np.asarray(ann_samples) / fs_mit, "label": ann_symbols})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 3.2
teach.check_step("Step 3.2", "answer", globals())

In [ ]:
#@title Ready code of step 3.2 (run it only if the code of the assistant does not pass the check)
data = pd.DataFrame({"time": np.asarray(ann_samples) / fs_mit, "label": ann_symbols})
beats = data[data["label"] != "+"]
labels = beats["label"].to_numpy()
rr_ms = np.diff(beats["time"].to_numpy()) * 1000
nn_ms = rr_ms[(labels[:-1] == "N") & (labels[1:] == "N")]
answer = round(np.std(nn_ms, ddof=1), 1)
print("NN intervals:", len(nn_ms), " mean (ms):", round(nn_ms.mean(), 1), " SDNN (ms):", answer)
teach.check_step("Step 3.2", "answer", globals())

**What the result shows.** The SDNN depends on the length of the recording, so a value of 30 minutes cannot be compared with one of 5 minutes or 24 hours. The standards of 1996 give separate reference values for recordings of 5 minutes and of 24 hours.

### Step 3.3: A real 12-lead ECG

**Goal.** Draw a real 12-lead ECG of a patient with an acute myocardial infarction as on ECG paper, and measure its heart rate.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to see the real 12-lead ECG of an 81-year-old woman admitted with an acute infero-lateral myocardial infarction in the layout of ECG paper, and to measure her heart rate.  
> Data: The notebook holds a table named "data". Each row is one moment of the recording: The column "time" gives the time in seconds, and the columns "I", "II", "III", "aVR", "aVL", "aVF" and "V1" to "V6" give the voltage of the twelve standard leads in millivolts. The recording lasts 38.4 seconds and was sampled 1000 times per second.  
> Task: Draw the first 3 seconds of the twelve leads in three rows and four columns, as on ECG paper: I, II and III in the first column, aVR, aVL and aVF in the second, V1 to V3 in the third and V4 to V6 in the fourth, with the name of the lead above each panel. Then find the R peaks of lead I. Accept as R peaks the peaks that exceed half of the highest value and lie at least 0.3 seconds apart. Compute the heart rate in beats per minute from the mean time between successive R peaks.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the heart rate with one decimal and keep it under the name "answer". Show the figure as well.  
> Test: The heart rate must lie between 40 and 150 beats per minute, and the number of R peaks must fit it: At 80 beats per minute, 38.4 seconds hold about 51 beats.  

In [ ]:
#@title Data of step 3.3: run this cell (it prepares the table "data" and shows it)
leads = {"i": "I", "ii": "II", "iii": "III", "avr": "aVR", "avl": "aVL", "avf": "aVF",
               "v1": "V1", "v2": "V2", "v3": "V3", "v4": "V4", "v5": "V5", "v6": "V6"}
data = ptb[list(leads)].rename(columns=leads)
data.insert(0, "time", np.arange(len(ptb)) / fs_ptb)
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 3.3
teach.check_step("Step 3.3", "answer", globals())

In [ ]:
#@title Ready code of step 3.3 (run it only if the code of the assistant does not pass the check)
leads = {"i": "I", "ii": "II", "iii": "III", "avr": "aVR", "avl": "aVL", "avf": "aVF",
               "v1": "V1", "v2": "V2", "v3": "V3", "v4": "V4", "v5": "V5", "v6": "V6"}
data = ptb[list(leads)].rename(columns=leads)
data.insert(0, "time", np.arange(len(ptb)) / fs_ptb)
from scipy.signal import find_peaks
t = data["time"].to_numpy()
layout = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
first3 = t < 3
fig, axes = plt.subplots(3, 4, figsize=(12, 6), sharex=True)
for ax, name in zip(axes.T.ravel(), layout):
    ax.plot(t[first3], data[name].to_numpy()[first3], color="k", lw=0.7)
    ax.set_title(name, fontsize=9)
plt.tight_layout()
plt.show()
x = data["I"].to_numpy()
r, _ = find_peaks(x, height=x.max() / 2, distance=int(0.3 * 1000))
answer = round(60 / np.diff(t[r]).mean(), 1)
print("R peaks:", len(r), " heart rate (bpm):", answer)
teach.check_step("Step 3.3", "answer", globals())

**What the result shows.** In the figure, look at the ST segments of the inferior leads II, III and aVF and of the lateral leads. An algorithm measures intervals; deciding on the pattern of an infarction is the work of the physician. The R peaks were counted in lead I, because in lead II of this recording the same simple rule misses more than a third of the beats: A rule must be checked in the lead to which it is applied.

### Step 3.4: From a careless prompt to the prompt card

**Goal.** From the short prompt "write a python function that gives the qtc from qt and rr in samples", an AI assistant wrote code that returned an impossible QTc of 17 ms. The data cell of this step shows this result. In this step, the same calculation is requested again with the seven fields of the prompt card, and the result is checked.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: On an ECG sampled 500 times per second, the QT interval and the RR interval were measured as numbers of samples. The cardiologist wants the QT interval corrected by Bazett's method, in milliseconds.  
> Data: The notebook holds a table named "data" with one row: The column "QT" gives the QT interval and the column "RR" the RR interval, both as numbers of samples. The recording was sampled 500 times per second.  
> Task: Turn the QT interval into milliseconds and the RR interval into seconds. Correct the QT by Bazett's method: Divide the QT in milliseconds by the square root of the RR interval in seconds.  
> Constraints: Use only this table. Do not assume the number of samples per second; use the 500 given here. The code must not read files, download anything from the internet or install programs.  
> Output: Print the corrected QT in milliseconds with one decimal and keep it under the name "answer".  
> Test: The same calculation must give exactly 400 ms for a QT of 200 samples and an RR interval of 500 samples, because the RR interval is then exactly 1 second.  

In [ ]:
#@title Data of step 3.4: run this cell (it prepares the table "data" and shows it)
data = pd.DataFrame({"QT": [196], "RR": [353]})
globals().pop("answer", None)

def qtc_of_the_short_prompt(qt_samples, rr_samples, fs=360):
    qt_ms = qt_samples / fs * 1000
    rr_ms = rr_samples / fs * 1000
    return qt_ms / rr_ms ** 0.5


print("The code of the short prompt gives a QTc of", round(qtc_of_the_short_prompt(196, 353), 1), "ms")
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 3.4
teach.check_step("Step 3.4", "answer", globals())

In [ ]:
#@title Ready code of step 3.4 (run it only if the code of the assistant does not pass the check)
data = pd.DataFrame({"QT": [196], "RR": [353]})
fs = 500
qt_ms = data["QT"].iloc[0] / fs * 1000      # QT in milliseconds
rr_s = data["RR"].iloc[0] / fs              # RR interval in seconds
assert abs((200 / fs * 1000) / (500 / fs) ** 0.5 - 400) < 1e-9   # test with a known answer
answer = round(qt_ms / rr_s ** 0.5, 1)
print("Corrected QT (ms):", answer)
teach.check_step("Step 3.4", "answer", globals())

**What the result shows.** Two silent errors made the first result impossible: The assistant assumed 360 samples per second, and it put the RR interval in milliseconds under the square root. The data field with its units, the constraint against an assumed sampling rate and the test with a known answer prevent both errors before the code is used for a patient.

**Stop 3 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#the-paper-of-the-week) and continue reading at «The paper of the week». The weekly project can also be done on the prompt path: Its prompts are in the section «Weekly project on the prompt path» below.

## Weekly project on the prompt path

*Project of the week: [README of Week 1, Weekly project](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-01/README.md#weekly-project)*

The weekly project can also be built with the prompts below, one for each step. Run the preparation cell once: It loads the data of the project. Each step then works like a step of the stops, with one difference: The project has no check cell and no ready code, because its results go into the report. The test of each prompt must hold before the next prompt is sent, and every prompt goes into the AI usage log.

In [ ]:
#@title Preparation of the project: run this cell once (it loads the data of the project)
from vclinic import measure as _M
from vclinic.datasets import read_physionet_annotation
clinic = vclinic.make_cohort(n=500, seed=RANDOM_STATE)
ann_samples, ann_symbols = read_physionet_annotation("100", "mitdb")
fs_mit = 360
project_patients = ["VC-0317", "VC-0079", "VC-0254", "VC-0188", "VC-0042",
                     "VC-0150", "VC-0007", "VC-0031", "VC-0018", "VC-0026"]


def _project_measure(e):
    """Heart rate, PR, QRS, QT and frontal axis of one ECG, with the net QRS voltages of leads I and aVF in every beat."""
    L, fs = e.leads.to_numpy(), e.fs
    r = _M.detect_r_peaks(L[:, 1], fs)
    on, off = _M.qrs_bounds(L, fs, r)
    x20 = _M.lowpass(L[:, 1], fs, 20)
    sinus = e.truth["rhythm"] == "sinus"
    pr, qrs, qt, net_i, net_avf, t_prev = [], [], [], [], [], None
    for k in range(len(r) - 1):
        b = max(0, on[k] - int(0.03 * fs))
        te = _M.t_end_tangent(x20, fs, off[k], r[k + 1], np.median(x20[b:on[k]]))
        qrs.append((off[k] - on[k]) / fs * 1000)
        qt.append((te - on[k]) / fs * 1000)
        if sinus and t_prev is not None and np.isfinite(t_prev):
            pr.append((on[k] - _M.p_onset(x20, fs, on[k], earliest=t_prev + 0.02 * fs)) / fs * 1000)
        t_prev = te
        for col, out in ((0, net_i), (5, net_avf)):
            seg = L[on[k]:off[k] + 1, col] - np.median(L[b:on[k], col])
            out.append(seg.max() + seg.min())
    axis = [vclinic.frontal_axis(a, v) for a, v in zip(net_i, net_avf)]
    return {"hr": 60 / (np.mean(np.diff(r)) / fs), "pr": np.nanmedian(pr) if np.isfinite(pr).any() else np.nan,
            "qrs": np.median(qrs), "qt": np.nanmedian(qt), "axis": float(np.median(axis)),
            "net_i": net_i, "net_avf": net_avf}


def _project_table(patients):
    rows = []
    for pid in patients:
        p = clinic.set_index("patient_id").loc[pid]
        e = vclinic.patient_ecg(clinic, pid)
        m, t = _project_measure(e), e.truth
        for name, key, true in [("heart rate", "hr", t["hr_bpm"]), ("PR", "pr", t["pr_ms"]), ("QRS", "qrs", t["qrs_ms"]),
                                ("QT", "qt", t["qt_ms"]), ("axis", "axis", t["axis_deg"])]:
            rows.append({"patient": pid, "rhythm": "sinus" if p["ecg_rhythm"] == "sinus" else "AF",
                         "conduction": p["ecg_conduction"], "measure": name,
                         "measured": round(float(m[key]), 1), "true": round(float(true), 1)})
    return pd.DataFrame(rows)


def _project_qt(patients):
    rows = []
    for pid in patients:
        p = clinic.set_index("patient_id").loc[pid]
        m = _project_measure(vclinic.patient_ecg(clinic, pid))
        rows.append({"patient": pid, "sex": "female" if p["sex"] == "F" else "male",
                     "heart rate": round(m["hr"], 1), "QT": round(m["qt"], 1)})
    return pd.DataFrame(rows)
print("The project is ready.")

### Project step 1: The frontal axis of the case

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to measure the frontal QRS axis of a 67-year-old woman from her resting ECG, beat by beat.  
> Data: The notebook holds a table named "data". Each row is a beat: The column "beat" gives its number, and the columns "I" and "aVF" give the net QRS voltage of these leads in millivolts. The net voltage is the largest positive plus the largest negative deflection of the QRS complex.  
> Task: For every beat, build the vector whose horizontal component is the voltage of lead I and whose vertical component is the voltage of lead aVF divided by 0.866. The frontal axis is the angle of this vector in degrees, measured from the direction of lead I, with positive angles towards the feet. Then take the median axis of all beats and print whether it lies in the normal range of −30 to +90 degrees.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the median axis in degrees with one decimal and keep it under the name "answer".  
> Test: A beat with 0.9 mV in lead I and −0.3 mV in lead aVF must give an axis of about −21 degrees. The registry gives an axis of 21 degrees for this patient, and the result should lie within about 10 degrees of it, because the net voltages also hold the small septal and terminal forces.  

In [ ]:
#@title Data of project step 1: run this cell (it prepares the table "data" and shows it)
_m = _project_measure(vclinic.patient_ecg(clinic, "VC-0317"))
data = pd.DataFrame({"beat": np.arange(1, len(_m["net_i"]) + 1),
                     "I": np.round(_m["net_i"], 3), "aVF": np.round(_m["net_avf"], 3)})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 2: Ten patients against the truth

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know how well an automatic ECG measurement agrees with the true values in ten patients of the virtual clinic, among them patients with atrial fibrillation or left bundle branch block.  
> Data: The notebook holds a table named "data". Each row is one measurement of one patient: The column "patient" gives the number of the patient, "rhythm" says "sinus" or "AF", and "conduction" says "normal" or "LBBB" (left bundle branch block). The column "measure" names the measurement ("heart rate", "PR", "QRS", "QT" or "axis"), "measured" gives the value of the automatic measurement and "true" the true value. Heart rates are in beats per minute, intervals in milliseconds and axes in degrees. There is no PR interval in atrial fibrillation, so these cells are empty.  
> Task: Compute the difference, measured minus true, for every row. For each measurement, print the mean absolute difference and the largest absolute difference with the patient in whom it occurs. Then show all differences in one table, with one row per patient and one column per measurement.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the two summaries and the table, and keep the mean absolute difference of the QT interval in milliseconds under the name "answer".  
> Test: The table must have ten rows. For patient VC-0317, the differences of the PR, QRS and QT intervals must be only a few milliseconds.  

In [ ]:
#@title Data of project step 2: run this cell (it prepares the table "data" and shows it)
# The ten patients of the project. The list may be changed; at least two of them need atrial fibrillation or a bundle branch block.
project_patients = ["VC-0317", "VC-0079", "VC-0254", "VC-0188", "VC-0042",
                     "VC-0150", "VC-0007", "VC-0031", "VC-0018", "VC-0026"]
data = _project_table(project_patients)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 3: The heart rate of record 100 over thirty minutes

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to see how the heart rate of a 69-year-old man changes over his 30-minute ambulatory ECG, record 100 of the MIT-BIH Arrhythmia Database.  
> Data: The notebook holds a table named "data". Each row is a mark that the cardiologists placed on the recording: The column "time" gives its place in seconds, and the column "label" its kind. "N" marks a normal beat, "A" an atrial premature beat and "V" a premature ventricular contraction; "+" marks a change of rhythm and is not a beat.  
> Task: Keep only the beats, that is, all labels except "+". For every beat after the first, compute the heart rate as 60 divided by the RR interval in seconds. Plot this heart rate against the time in minutes. Then compute the mean heart rate of the whole recording as 60 divided by the mean RR interval.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the mean heart rate in beats per minute with one decimal and keep it under the name "answer". Show the plot as well.  
> Test: An RR interval of exactly 1 second must give 60 beats per minute, and the mean heart rate must lie between 60 and 90 beats per minute.  

In [ ]:
#@title Data of project step 3: run this cell (it prepares the table "data" and shows it)
data = pd.DataFrame({"time": np.asarray(ann_samples) / fs_mit, "label": ann_symbols})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 4: Bazett or Fridericia for each patient

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: For ten patients of the virtual clinic, the cardiologist wants to know whether the choice between Bazett's and Fridericia's correction changes the decision that the QT interval is prolonged.  
> Data: The notebook holds a table named "data". Each row is a patient: The column "patient" gives the number of the patient, and "sex" says "female" or "male". The column "heart rate" gives the measured heart rate in beats per minute, and "QT" the measured QT interval in milliseconds.  
> Task: For every patient, compute the RR interval in seconds as 60 divided by the heart rate. Compute the QTc by Bazett's formula, the QT divided by the square root of RR, and by Fridericia's formula, the QT divided by the cube root of RR. Call a QTc prolonged when it is 460 ms or longer in a woman, or 450 ms or longer in a man. List the patients for whom the two formulas lead to different decisions.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print a table with the two corrections and the two decisions, and the list of the disagreements. Keep the number of patients with different decisions under the name "answer".  
> Test: Patient VC-0317, a woman at about 85 beats per minute, must be prolonged by Bazett's formula and not by Fridericia's.  

In [ ]:
#@title Data of project step 4: run this cell (it prepares the table "data" and shows it)
data = _project_qt(project_patients)
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

<a id="part-2"></a>

# Part 2: Python path

**For whom?** This part is for participants who want to learn to write Python. Part 1 is not needed for it: After Section 0, start here.

**How it works.** Run the cells from top to bottom with Shift+Enter. The *Python step* at the start teaches the parts of the language that the week needs. In the numbered sections, exercises print *correct* or *not yet* at once, and their reference solutions sit in collapsed cells: Open them only after trying. *Observe and reflect* cells ask open questions and have no solution cell. *Python note* cells explain the language where a new construct first appears. Cells marked *Advanced (optional)* are extensions for those who already program. A participant who opens this part in the middle runs the *Catch-up* cell at the start of that section. Some exercise cells begin with a line that prepares a table named `data`: It serves the prompt of the exercise in the prompt workshop of the interactive lab, and on the Python path the exercise can be solved with the tables named in its text.

## Python step 1: Variables, lists and loops with medical examples; dictionaries, decisions and functions

*Lecture: [Variables, lists and loops with medical examples](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#variables-lists-and-loops-with-medical-examples)*

Python is the language of this course. A code cell holds Python instructions, and Shift+Enter runs them. Text after `#` is a comment: Python skips it, and it explains the code to the reader. Every cell of this step begins with the line `# Python practice: a small example for learning the language`. The examples use the numbers of the case of the week, a 67-year-old woman with a heart rate of 85 beats per minute (bpm) and a QT interval of 391 ms.

In [ ]:
# Python practice: a small example for learning the language
heart_rate = 85          # beats per minute (bpm)
qt_ms = 391              # QT interval in milliseconds
rr_s = 60 / heart_rate   # RR interval in seconds
print(rr_s)
print(round(rr_s, 3))

**Python note.** `heart_rate = 85` creates a *variable*: a name that holds a value. The sign `=` assigns; it does not test equality. `/` divides, and `60 / heart_rate` gives the RR interval in seconds. `print()` shows a value, and `round(value, 3)` rounds it to three decimals. Python counts in floating-point numbers, so `0.7058823529411765` is printed in full until it is rounded.

In [ ]:
# Python practice: a small example for learning the language
patient_id = "VC-0317"
print(f"Patient {patient_id}: {heart_rate} bpm, RR {rr_s:.3f} s")

**Python note.** Text in quotes is a *string*. A string that starts with `f` is an *f-string*: Values inside curly braces are put into the text, and `{rr_s:.3f}` writes the number with three decimals.

In [ ]:
# Python practice: a small example for learning the language
troponin = [3.6, 4.1, 3.9]          # serial measurements (ng/L)
print(troponin[0], troponin[-1], len(troponin))
troponin.append(4.0)                # a fourth measurement arrives
print(troponin, "max:", max(troponin))

**Python note.** Square brackets make a *list*, an ordered series of values. Python counts positions from 0, so `troponin[0]` is the first value and `troponin[-1]` the last. `len()` counts the elements, `.append()` adds one at the end, and `max()` returns the largest.

In [ ]:
# Python practice: a small example for learning the language
patient = {"id": "VC-0317", "age": 67, "sex": "F", "hr_bpm": 85, "qt_ms": 391}
print(patient["age"])
patient["lvef"] = 63                # a new field: left ventricular ejection fraction (%)
print(patient)

**Python note.** Curly braces with `key: value` pairs make a *dictionary*, which works like a patient chart: Each field has a label. `patient["age"]` reads a field, and `patient["lvef"] = 63` adds one.

In [ ]:
# Python practice: a small example for learning the language
qtc_ms = 465
limit = 460 if patient["sex"] == "F" else 450
if qtc_ms >= limit:
    print(f"QTc {qtc_ms} ms is at or above the limit of {limit} ms")
else:
    print(f"QTc {qtc_ms} ms is below the limit of {limit} ms")

**Python note.** `if` runs the indented lines below it only when its condition is true, and `else` covers the other case. The indentation (four spaces) marks which lines belong to each branch. `==` tests equality and `>=` means "greater than or equal". The line `limit = 460 if ... else 450` chooses one of two values in a single line. The two limits are the thresholds of a prolonged QTc for women and men in the 2009 recommendations of the American Heart Association (AHA) and its partners.

In [ ]:
# Python practice: a small example for learning the language
clinic = [
    {"id": "VC-0317", "hr": 85, "qt": 391},
    {"id": "VC-0079", "hr": 58, "qt": 404},
    {"id": "VC-0254", "hr": 98, "qt": 343},
]
for p in clinic:
    rr = 60 / p["hr"]
    print(p["id"], round(rr, 3))

**Python note.** `clinic` is a list of dictionaries, one per patient. `for p in clinic:` takes the patients one by one, and the indented lines run once for each of them, like a ward round.

In [ ]:
# Python practice: a small example for learning the language
def qtc_bazett(qt_ms, rr_s):
    """QT interval corrected for heart rate with Bazett's formula (ms)."""
    return qt_ms / rr_s ** 0.5


print(round(qtc_bazett(391, 60 / 85), 1))

**Python note.** `def` defines a *function*, a named procedure that can be used again, much like a clinical protocol. `qt_ms` and `rr_s` are its inputs, and `return` gives back the result. The text in triple quotes is the docstring, the short description of the function. `**` raises to a power, so `rr_s ** 0.5` is the square root.

In [ ]:
# Python practice: a small example for learning the language
rr = np.array([0.81, 0.85, 0.83])   # three RR intervals (s)
print(60 / rr)                      # heart rate of each beat (bpm)
print(rr.mean(), rr.std(ddof=1))

**Python note.** NumPy, imported in Section 0 as `np`, works on whole arrays of numbers at once: `60 / rr` divides 60 by each RR interval. `.mean()` returns the mean, and `.std(ddof=1)` the sample standard deviation, which divides by $n-1$.

**Quick check.** After `troponin.append(4.0)`, what does `troponin[-1]` return?

<details><summary>Answer</summary><code>4.0</code>: the index <code>-1</code> always returns the last element, and <code>append()</code> put <code>4.0</code> at the end.</details>

**Exercise 1.1.** Compute the RR interval in seconds for a heart rate of 72 bpm and round it to three decimals.

In [ ]:
answer = None   # your answer
teach.check("Exercise 1.1", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(60 / 72, 3)
teach.check("Exercise 1.1", answer)

**Exercise 1.2.** Write a function `qtc_fridericia(qt_ms, rr_s)` that divides the QT interval by the cube root of the RR interval, $QT_{cF} = QT / RR^{1/3}$. Use it for the case of the week (QT 391 ms, heart rate 85 bpm) and round to one decimal.

In [ ]:
def qtc_fridericia(qt_ms, rr_s):
    return None   # replace None with the formula


answer = qtc_fridericia(391, 60 / 85)
answer = round(answer, 1) if answer is not None else None
teach.check("Exercise 1.2", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def qtc_fridericia(qt_ms, rr_s):
    """QT interval corrected with Fridericia's formula (ms)."""
    return qt_ms / rr_s ** (1 / 3)


answer = round(qtc_fridericia(391, 60 / 85), 1)
teach.check("Exercise 1.2", answer)

**Exercise 1.3.** Count the patients of the list `clinic` whose heart rate is above 80 bpm. Use a `for` loop and an `if`.

In [ ]:
count = 0
# your loop here
answer = None   # your answer
teach.check("Exercise 1.3", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
count = 0
for p in clinic:
    if p["hr"] > 80:
        count = count + 1
answer = count
teach.check("Exercise 1.3", answer)

## 1. Mini project: The Kaggle Heart Disease data set: loading with pandas, statistical summary and visualisation

*Lecture: [Mini project: Loading the Kaggle Heart Disease data set, statistical summary and visualisation](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#mini-project-loading-the-kaggle-heart-disease-data-set-statistical-summary-and-visualisation)*

The Heart Disease data set shared on Kaggle comes from the Cleveland data of the UCI Heart Disease collection: 14 variables of 303 patients who underwent coronary angiography. The column `num` grades the disease from 0 (no narrowing of more than 50 percent) to 4. The course bundles the original file of the UCI repository, so no Kaggle account is needed. This section is the in-class mini project of the week: Load the data with pandas, process them, summarise them with statistics and visualise them with Matplotlib.

**From the formulas to the code.** The mean $\bar{x}$ is `df["chol"].mean()`. The sample standard deviation $s$, with $n-1$ in the denominator, is `df["chol"].std()`. The median is `.median()`, and the interquartile range is `.quantile(0.75) - .quantile(0.25)`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 1 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB01_python_and_ecg_basics.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(1)

In [ ]:
from vclinic.datasets import load_cleveland

heart = load_cleveland()
print(heart.shape)          # (rows, columns)
heart.head()

**Python note.** pandas, imported as `pd`, keeps a table in a *DataFrame*: rows are patients and columns are variables. `load_cleveland()` reads the bundled file with `pd.read_csv` and names the columns. `.shape` gives the number of rows and columns, and `.head()` shows the first five rows.

| Column | Meaning |
|---|---|
| `age`, `sex` | Age in years; sex (1 = male, 0 = female) |
| `cp` | Chest pain type (1 typical angina, 2 atypical angina, 3 non-anginal pain, 4 asymptomatic) |
| `trestbps` | Resting systolic blood pressure (mmHg) |
| `chol` | Serum cholesterol (mg/dL) |
| `fbs` | Fasting blood sugar above 120 mg/dL (1 = yes) |
| `restecg` | Resting ECG (0 normal, 1 ST-T abnormality, 2 left ventricular hypertrophy) |
| `thalach` | Maximum heart rate reached in the exercise test (bpm) |
| `exang` | Exercise-induced angina (1 = yes) |
| `oldpeak` | ST depression induced by exercise relative to rest (mm) |
| `slope` | Slope of the peak exercise ST segment (1 up, 2 flat, 3 down) |
| `ca` | Number of major vessels coloured by fluoroscopy (0 to 3) |
| `thal` | Thallium scan (3 normal, 6 fixed defect, 7 reversible defect) |
| `num`, `disease` | Angiographic disease grade 0 to 4; `disease` = 1 when `num` > 0 |

In [ ]:
# The worked example of the lecture: five cholesterol values
chol5 = heart["chol"].head(5)
print(list(chol5))
mean = chol5.sum() / len(chol5)
sd = (((chol5 - mean) ** 2).sum() / (len(chol5) - 1)) ** 0.5
print(f"mean by hand: {mean:.2f}  pandas: {chol5.mean():.2f}")
print(f"SD by hand: {sd:.2f}  pandas: {chol5.std():.2f}")
print(f"median: {chol5.median():.1f}")

In [ ]:
# Summary of the main variables, by disease status
cols = ["age", "trestbps", "chol", "thalach", "oldpeak"]
print(heart["disease"].value_counts())
heart.groupby("disease")[cols].agg(["mean", "std"]).round(1)

**Python note.** `value_counts()` counts each value of a column. `groupby("disease")` splits the table into the patients with and without disease, and `.agg(["mean", "std"])` computes the two statistics for each group and each column.

In [ ]:
# Processing the table: missing values, a selection, a new column and a sorted view
print(heart.isna().sum()[heart.isna().sum() > 0])        # missing values per column
older = heart[heart["age"] >= 60]                          # rows that meet a condition
print("patients aged 60 or more:", len(older))
heart["chol_mmol"] = (heart["chol"] / 38.67).round(2)      # a new column: cholesterol in mmol/L
heart.sort_values("thalach")[["age", "sex", "thalach", "disease"]].head()

**Python note.** `heart.isna().sum()` counts the missing values of every column. `heart[heart["age"] >= 60]` keeps the rows for which the condition is true. A new column is created by assignment, here the total cholesterol converted from mg/dL to mmol/L by dividing by 38.67. `sort_values("thalach")` sorts the rows by the maximum heart rate, and a list of names in double brackets selects columns.

**A first cardiological plot with Matplotlib.** *Lecture: [A first cardiological plot with Matplotlib](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#a-first-cardiological-plot-with-matplotlib)*

`plt.subplots(1, 2)` creates a figure with two sets of axes side by side. `hist` draws a histogram of the age in each group, and `boxplot` compares the maximum heart rate of the two groups. Every axis names its quantity and its unit.

In [ ]:
# Two first figures of the registry
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for value, color, label in [(0, "#1F5F8B", "no disease"),
                            (1, "#9B1B1B", "disease")]:
    sub = heart[heart["disease"] == value]
    axes[0].hist(sub["age"], bins=15, alpha=0.6, color=color, label=label)
axes[0].set_xlabel("Age (years)")
axes[0].set_ylabel("Number of patients")
axes[0].legend()
heart.boxplot(column="thalach", by="disease", ax=axes[1])
axes[1].set_title("")
axes[1].set_xlabel("Disease (0 no, 1 yes)")
axes[1].set_ylabel("Maximum heart rate (bpm)")
fig.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
# The same comparison with Seaborn, which builds on Matplotlib
try:
    import seaborn as sns
    fig, ax = plt.subplots(figsize=(5.5, 4))
    sns.boxplot(data=heart, x="disease", y="thalach", color="#C9A0A0", ax=ax)
    ax.set_xlabel("Disease (0 no, 1 yes)")
    ax.set_ylabel("Maximum heart rate (bpm)")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Seaborn is not installed; the Matplotlib figure above shows the same comparison.")

**Exercise A.** What is the mean age of the patients with disease (`disease == 1`)? Round to one decimal.

In [ ]:
# The table "data" for the prompt of this exercise (Part F of the interactive lab)
data = pd.DataFrame({"age": heart["age"],
                     "CAD": heart["disease"].map({1: "yes", 0: "no"})})
answer = None   # your answer
teach.check("Exercise A", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(heart.loc[heart["disease"] == 1, "age"].mean(), 1)
teach.check("Exercise A", answer)

**Exercise B.** What percentage of the 303 patients are women (`sex == 0`)? Round to one decimal.

In [ ]:
# The table "data" for the prompt of this exercise (Part F of the interactive lab)
data = pd.DataFrame({"age": heart["age"],
                     "sex": heart["sex"].map({1: "male", 0: "female"})})
answer = None   # your answer
teach.check("Exercise B", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(100 * (heart["sex"] == 0).mean(), 1)
teach.check("Exercise B", answer)

**Observe and reflect.** Which variable differs most between the two groups in the summary table? Is the direction of the difference clinically plausible? What does the small number of women tell about how far the data can be generalised?

**Advanced (optional).** Two columns hold missing values. Count them with `heart.isna().sum()`, then compare the disease rate by chest pain type with `pd.crosstab(heart["cp"], heart["disease"], normalize="index")`.

In [ ]:
# Advanced (optional)
print(heart.isna().sum()[heart.isna().sum() > 0])
pd.crosstab(heart["cp"], heart["disease"], normalize="index").round(2)

## 2. The virtual clinic opens

*Lecture: [The virtual cardiology clinic, the project of the course](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#the-virtual-cardiology-clinic-the-project-of-the-course)*

`vclinic.make_cohort()` creates the registry of the virtual cardiology clinic. Each patient has demographic data, risk factors, laboratory values, a left ventricular ejection fraction (LVEF), a main diagnosis and the parameters of a resting 12-lead electrocardiogram (ECG). A patient keeps the same values in every week of the course. The data are synthetic: They follow broad clinical patterns but describe no real person.

**Started here?** Run the catch-up cell below once. It runs the code that Section 2 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB01_python_and_ecg_basics.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(2)

In [ ]:
clinic = vclinic.make_cohort(n=500, seed=2026)
print(clinic.shape)
clinic.head()

**Python note.** `vclinic.make_cohort(n=500, seed=2026)` calls a function of the course package with two *keyword arguments*. The seed fixes the random numbers, so every run gives the same 500 patients.

In [ ]:
# Main diagnoses and rhythm of the clinic
print(clinic["main_diagnosis"].value_counts())
print(clinic["ecg_rhythm"].value_counts())
clinic[["age", "bmi", "sbp", "ldl", "hdl", "egfr", "lvef", "ntprobnp",
        "ecg_hr", "ecg_pr", "ecg_qrs", "ecg_qt"]].describe().round(1)

The diagnosis codes are the international abbreviations: `CCS` chronic coronary syndrome, `ACS` acute coronary syndrome, `HFrEF` heart failure with reduced ejection fraction, `HFpEF` heart failure with preserved ejection fraction and `none` no structural heart disease. `AF` in the rhythm column is atrial fibrillation. The full data dictionary is in DATASETS.md.

In [ ]:
# The virtual clinic next to the Cleveland data
compare = pd.DataFrame({
    "Cleveland (real)": [
        heart["age"].mean(), 100 * heart["sex"].mean(),
        heart["trestbps"].mean(), heart["chol"].mean(),
        100 * heart["disease"].mean()],
    "Virtual clinic (synthetic)": [
        clinic["age"].mean(), 100 * (clinic["sex"] == "M").mean(),
        clinic["sbp"].mean(), clinic["total_chol"].mean(),
        100 * clinic["cad"].mean()],
}, index=["Mean age (years)", "Men (%)",
          "Systolic pressure (mmHg)",
          "Total cholesterol (mg/dL)",
          "Coronary disease (%)"])
compare.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.hist(heart["age"], bins=20, density=True, alpha=0.55, color="#1F5F8B",
        label="Cleveland (real)")
ax.hist(clinic["age"], bins=20, density=True, alpha=0.55, color="#9B1B1B",
        label="Virtual clinic (synthetic)")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.show()

**Exercise C.** How many patients of the virtual clinic have the main diagnosis `HFrEF`?

In [ ]:
# The table "data" for the prompt of this exercise (Part F of the interactive lab)
data = pd.DataFrame({"patient": clinic["patient_id"],
                     "main diagnosis": clinic["main_diagnosis"].replace({"none": "none"}),
                     "LVEF": clinic["lvef"]})
answer = None   # your answer
teach.check("Exercise C", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int((clinic["main_diagnosis"] == "HFrEF").sum())
teach.check("Exercise C", answer)

**Observe and reflect.** The two cohorts look alike in the table. In which ways could a synthetic registry still be wrong in a way that matters for a model trained on it? Think of rare conditions, of correlations between variables and of measurement noise.

**The Python path of Practice stop 1 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#last-two-hours-working-with-ecg-data) and continue reading at «Last two hours: Working with ECG data».

## 3. The ECG as data: Samples, sampling frequency and units

*Lecture: [The ECG as data](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#the-ecg-as-data)*

**From the formulas to the code.** The sampling frequency $f_s$ is `case.fs`. The time of sample $n$, $t_n = n / f_s$, is in `case.time`. The number of samples in an interval, $N = \Delta t \cdot f_s$, is `round(dt * case.fs)`. The voltages in millivolts are the columns of `case.leads`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 3 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB01_python_and_ecg_basics.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(3)

In [ ]:
case = vclinic.patient_ecg(clinic, "VC-0317")   # the case of the week
print(case)
print("samples per lead:", len(case.leads))
print("time step (s):", 1 / case.fs)
case.leads.head()

In [ ]:
# Lead II on ECG paper: 10 seconds and one beat
fig, axes = plt.subplots(2, 1, figsize=(12, 6.5))
teach.plot_strip(axes[0], case.time, case.lead("II"), label="II")
one = (case.time >= 1.0) & (case.time <= 2.0)
teach.plot_strip(axes[1], case.time[one], case.lead("II")[one], label="II")
axes[1].plot(case.time[one][::10], case.lead("II")[one][::10], "o",
             color="#9B1B1B", ms=3, label="every 10th sample")
axes[1].legend(loc="lower right")
plt.tight_layout()
plt.show()

**Python note.** `case.time >= 1.0` compares every time value with 1.0 and returns an array of `True` and `False`, a *mask*. Two masks joined with `&` keep the samples between 1 and 2 seconds, and `x[mask]` selects them. `[::10]` takes every tenth element.

In [ ]:
# Samples in one RR interval at three sampling frequencies
rr = 60 / 85
for fs in [500, 360, 1000]:
    print(f"fs = {fs} Hz: RR = {rr:.3f} s spans {round(rr * fs)} samples")

**Explore.** The panel samples one heartbeat of the case again at a lower frequency and stores it with fewer bits, as an analog-to-digital converter (ADC) would do. The reference is the same beat at 2000 Hz. Move the sliders and watch how the height and the time of the R peak change. Below about 100 Hz the R peak is clipped and shifted. With few bits the trace turns into steps.

In [ ]:
# Explore: sampling frequency and resolution
fine = vclinic.simulate(case.params, duration_s=2.0, fs=2000, noise="none", seed=7)
t_ref, x_ref = fine.time, fine.lead("II")


def show_sampling(fs_hz=500, bits=12):
    step = int(round(2000 / fs_hz))
    t_s, x_s = t_ref[::step], x_ref[::step]
    q = 10.0 / 2 ** bits                       # step size over a 10 mV range
    x_q = np.round(x_s / q) * q
    window = (t_ref > 0.25) & (t_ref < 0.75)
    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.plot(t_ref[window], x_ref[window], color="#888888", lw=1,
            label="2000 Hz reference")
    w2 = (t_s > 0.25) & (t_s < 0.75)
    ax.step(t_s[w2], x_q[w2], where="mid", color="#9B1B1B", lw=1.2,
            label=f"{fs_hz} Hz, {bits} bit")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("mV")
    ax.legend(loc="upper right")
    plt.show()
    print(f"R peak: reference {x_ref.max():.3f} mV, "
          f"sampled {x_q.max():.3f} mV; "
          f"time shift {abs(t_s[np.argmax(x_q)] - t_ref[np.argmax(x_ref)]) * 1000:.1f} ms; "
          f"resolution {q * 1000:.2f} µV")


try:
    from ipywidgets import interact, IntSlider
    interact(show_sampling,
             fs_hz=IntSlider(value=500, min=50, max=1000, step=50, description="fs (Hz)"),
             bits=IntSlider(value=12, min=4, max=16, step=1, description="bits"))
except ImportError:
    show_sampling(500, 12)
    show_sampling(100, 6)

**Exercise D.** On ECG paper one large box lasts 0.2 s. How many samples does one large box hold at 500 Hz?

In [ ]:
answer = None   # your answer
teach.check("Exercise D", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(0.2 * 500)
teach.check("Exercise D", answer)

## 4. Twelve leads from one heart vector

*Lecture: [The heart as a vector: Twelve leads, one source](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#the-heart-as-a-vector-twelve-leads-one-source)*

**From the formulas to the code.** The heart vector $\mathbf{H} = (H_x, H_y, H_z)$ is a NumPy array of three numbers. The lead vectors $\mathbf{L}$ of the 12 leads are the rows of `vclinic.LEAD_MATRIX`. The lead voltage $V_L = \mathbf{L} \cdot \mathbf{H}$ for all leads at once is `vclinic.project(H)`, which computes `LEAD_MATRIX @ H`. The frontal axis $\alpha = \operatorname{atan2}(H_y, H_x)$ is `vclinic.frontal_axis(V_I, V_aVF)`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 4 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB01_python_and_ecg_basics.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(4)

In [ ]:
# The lead vectors of the virtual clinic: x to the left, y down, z to the front
lead_table = pd.DataFrame(vclinic.LEAD_MATRIX, index=vclinic.LEADS,
                          columns=["Lx", "Ly", "Lz"]).round(3)
lead_table

In [ ]:
# The worked example of the lecture
H = np.array([0.6, 0.8, -0.2])           # heart vector (mV)
V = vclinic.project(H)
volts = pd.Series(V, index=vclinic.LEADS).round(3)
print(volts.to_string())
print("Einthoven: I + III =", round(volts["I"] + volts["III"], 3), "and II =", volts["II"])
print("Goldberger: aVR + aVL + aVF =", round(volts["aVR"] + volts["aVL"] + volts["aVF"], 3))
print("Frontal axis (degrees):", round(vclinic.frontal_axis(V[0], V[5]), 1))

**Python note.** `@` multiplies a matrix by a vector. Each row of the 12 x 3 lead matrix is multiplied element by element with the three numbers of `H`, and the products are added. A `pd.Series` is a single labelled column, here indexed by the lead names.

In [ ]:
# The 12-lead ECG of the case
fig = teach.plot_12lead(case.time, case.leads, title="VC-0317")
plt.show()
L = case.leads
print("Largest deviation from Einthoven's law (mV):",
      float(np.abs(L["II"] - (L["I"] + L["III"])).max()))

**Explore.** Turn the heart vector in the frontal plane and toward the front or the back of the chest, and watch the twelve projections. A lead shows its largest positive deflection when the vector points along it and a negative one when the vector points away.

In [ ]:
# Explore: one vector, twelve projections
def show_projection(frontal=50, anterior=-15, size=1.3):
    h = size * vclinic.direction(frontal, anterior)
    v = vclinic.project(h)
    colors = ["#9B1B1B" if x >= 0 else "#1F5F8B" for x in v]
    fig, ax = plt.subplots(figsize=(9, 3.4))
    ax.bar(vclinic.LEADS, v, color=colors)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_ylim(-2.5, 2.5)
    ax.set_ylabel("Projection (mV)")
    ax.set_title(f"H = ({h[0]:.2f}, {h[1]:.2f}, {h[2]:.2f}) mV, axis "
                 f"{vclinic.frontal_axis(v[0], v[5]):.0f}°")
    plt.show()


try:
    from ipywidgets import interact, IntSlider, FloatSlider
    interact(show_projection,
             frontal=IntSlider(value=50, min=-180, max=180, step=5, description="frontal (°)"),
             anterior=IntSlider(value=-15, min=-90, max=90, step=5, description="front/back (°)"),
             size=FloatSlider(value=1.3, min=0.2, max=2.0, step=0.1, description="size (mV)"))
except ImportError:
    show_projection(50, -15, 1.3)

**Exercise E.** Compute the voltage of lead aVL for $\mathbf{H} = (0.6, 0.8, -0.2)$ mV with the lead vector of aVL from `lead_table`, and round to three decimals.

In [ ]:
answer = None   # your answer
teach.check("Exercise E", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
l_avl = vclinic.LEAD_VECTORS["aVL"]
answer = round(float(l_avl @ np.array([0.6, 0.8, -0.2])), 3)
teach.check("Exercise E", answer)

**Observe and reflect.** The same vector gives a positive deflection in lead II and a negative one in lead V1. Explain the difference with the angle between the vector and each lead. Why do the chest leads show larger deflections than the limb leads for vectors of the same size?

## 5. Measuring PR, QRS and QT

*Lecture: [Intervals: PR, QRS and QT](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#intervals-pr-qrs-and-qt)*

**From the formulas to the code.** The heart rate $HR = 60 / RR$ is `60 / rr_s`. Bazett's correction $QT_{cB} = QT / \sqrt{RR}$ is `qt / np.sqrt(rr_s)`, and Fridericia's $QT_{cF} = QT / \sqrt[3]{RR}$ is `qt / np.cbrt(rr_s)`; both need the RR interval in seconds. The end of the T wave by the tangent method, $t_{end} = t_s + (b - x(t_s)) / x'(t_s)$, is the line `t_s + (base - x[t_s]) / slope * fs` in the function `t_end_tangent`.

The section builds the measurement in five steps and then joins them in one function, `measure_ecg`. The same steps are kept in `vclinic.measure` for the later weeks.

**Started here?** Run the catch-up cell below once. It runs the code that Section 5 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB01_python_and_ecg_basics.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(5)

In [ ]:
# Step 1. R peaks: band-pass, derivative, squaring and a 150 ms moving window
from scipy.signal import butter, filtfilt, find_peaks


def bandpass(x, fs, low, high):
    b, a = butter(2, [low / (fs / 2), high / (fs / 2)], btype="band")
    return filtfilt(b, a, x)


def lowpass(x, fs, cutoff=20.0):
    b, a = butter(2, cutoff / (fs / 2), btype="low")
    return filtfilt(b, a, x)


def detect_r_peaks(x, fs):
    y = bandpass(x, fs, 5, 15)                 # keeps the QRS frequencies
    d = np.gradient(y) * fs                    # slope (mV/s)
    z = d ** 2                                 # only large slopes stand out
    w = int(round(0.150 * fs))                 # 150 ms window
    m = np.convolve(z, np.ones(w) / w, mode="same")
    peaks, _ = find_peaks(m, height=0.3 * np.percentile(m, 99),
                          distance=int(0.25 * fs))
    yy = bandpass(x, fs, 1, 40)
    half = int(0.075 * fs)
    r = []
    for p in peaks:                            # largest deflection within 75 ms
        lo, hi = max(0, p - half), min(len(x), p + half + 1)
        r.append(lo + int(np.argmax(np.abs(yy[lo:hi]))))
    return np.unique(r)


r_idx = detect_r_peaks(case.lead("II"), case.fs)
print("R peaks found:", len(r_idx))
print("times (s):", np.round(r_idx / case.fs, 3))
print("true R times (s):", np.round(case.truth["all_beats_s"], 3))

In [ ]:
# Step 2. QRS onset and offset from the slope of all 12 leads
def global_slope(leads, fs):
    arr = np.column_stack([bandpass(leads[:, j], fs, 1, 40)
                           for j in range(leads.shape[1])])
    g = np.abs(np.gradient(arr, axis=0)).sum(axis=1) * fs
    w = max(1, int(round(0.010 * fs)))
    return np.convolve(g, np.ones(w) / w, mode="same")


def qrs_bounds(leads, fs, r_idx, frac=0.08):
    g = global_slope(leads, fs)
    onsets, offsets = [], []
    for r in r_idx:
        lo, hi = max(0, r - int(0.12 * fs)), min(len(g) - 1, r + int(0.14 * fs))
        level = frac * g[lo:hi + 1].max()      # 8 percent of the largest slope
        on = r
        while on > lo and g[on] > level:
            on -= 1
        off = r
        while off < hi and g[off] > level:
            off += 1
        onsets.append(on)
        offsets.append(off)
    return np.array(onsets), np.array(offsets)


on_idx, off_idx = qrs_bounds(case.leads.to_numpy(), case.fs, r_idx)
qrs_ms = (off_idx - on_idx) / case.fs * 1000
print("QRS durations (ms):", np.round(qrs_ms, 0))

**Python note.** `while` repeats its indented lines as long as its condition holds. Here it walks backward from the R peak, one sample at a time, until the slope of the 12 leads falls below the level. `np.column_stack` places arrays side by side as the columns of a matrix.

In [ ]:
# Step 3. End of the T wave by the tangent method (lead II, low-passed at 20 Hz)
def t_end_tangent(x, fs, qrs_off, next_r, base):
    lo = qrs_off + int(0.08 * fs)
    hi = qrs_off + int(0.7 * (next_r - qrs_off))
    seg = x[lo:hi] - base
    k = int(np.argmax(np.abs(seg)))
    peak = lo + k                              # T peak
    sign = 1.0 if seg[k] >= 0 else -1.0
    d = np.gradient(x) * fs
    stop = min(len(x) - 1, peak + int(0.20 * fs))
    t_s = peak + int(np.argmin(sign * d[peak:stop]))   # steepest point
    slope = d[t_s]
    if slope == 0:
        return np.nan
    return t_s + (base - x[t_s]) / slope * fs  # where the tangent meets the baseline


x20 = lowpass(case.lead("II"), case.fs, 20)
fs = case.fs
i = 2                                          # third beat
base = np.median(x20[on_idx[i] - int(0.03 * fs):on_idx[i]])
t_end = t_end_tangent(x20, fs, off_idx[i], r_idx[i + 1], base)
print(f"QT of beat: {(t_end - on_idx[i]) / fs * 1000:.1f} ms, true: {case.truth['qt_ms']:.0f} ms")

In [ ]:
# The tangent construction on the third beat
d20 = np.gradient(x20) * fs
seg = x20[off_idx[i] + int(0.08 * fs):off_idx[i] + int(0.7 * (r_idx[i + 1] - off_idx[i]))]
peak = off_idx[i] + int(0.08 * fs) + int(np.argmax(np.abs(seg - base)))
t_s = peak + int(np.argmin(d20[peak:peak + int(0.2 * fs)]))
tt = np.linspace(t_s / fs - 0.05, t_end / fs + 0.02, 50)
fig, ax = plt.subplots(figsize=(9, 4))
w = slice(on_idx[i] - int(0.1 * fs), int(t_end) + int(0.12 * fs))
teach.plot_strip(ax, case.time[w], x20[w], label="II", v_range=(-0.5, 1.5))
ax.plot(tt, x20[t_s] + d20[t_s] * (tt - t_s / fs), color="#9B1B1B", lw=1.5,
        label="tangent at the steepest point")
ax.axhline(base, color="#1F5F8B", lw=1, ls="--", label="baseline")
ax.axvline(on_idx[i] / fs, color="k", lw=1, ls=":")
ax.axvline(t_end / fs, color="#9B1B1B", lw=1, ls=":")
ax.legend(loc="upper right", fontsize=8)
plt.show()

In [ ]:
# Step 4. Onset of the P wave
def p_onset(x, fs, qrs_on, earliest=0, frac=0.10):
    lo = max(0, int(earliest), qrs_on - int(0.30 * fs))
    hi = qrs_on - int(0.04 * fs)
    if hi - lo < int(0.08 * fs):               # too short to hold a P wave
        return np.nan
    seg = x[lo:hi]
    base = np.median(seg[:int(0.04 * fs)])     # flat segment before the P wave
    seg = seg - base
    k = int(np.argmax(seg))
    j = k
    while j > 0 and seg[j] > frac * seg[k]:    # back to 10 percent of the P height
        j -= 1
    return lo + j


po = p_onset(x20, fs, on_idx[i], earliest=t_end_tangent(
    x20, fs, off_idx[i - 1], r_idx[i], base) + 0.02 * fs)
print(f"PR of beat: {(on_idx[i] - po) / fs * 1000:.0f} ms, true: {case.truth['pr_ms']:.0f} ms")

In [ ]:
# Step 5. All steps in one function: the measurement of the ECG laboratory
def measure_ecg(leads, fs, lead="II", sinus=True):
    """Median heart rate, PR, QRS, QT and QTc of a 12-lead recording."""
    leads = np.asarray(leads, dtype=float)
    x = leads[:, ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3",
                  "V4", "V5", "V6"].index(lead)]
    r = detect_r_peaks(leads[:, 1], fs)
    on, off = qrs_bounds(leads, fs, r)
    x20 = lowpass(x, fs, 20)
    pr, qrs, qt = [], [], []
    t_prev = None
    for k in range(len(r) - 1):
        base = np.median(x20[max(0, on[k] - int(0.03 * fs)):on[k]])
        te = t_end_tangent(x20, fs, off[k], r[k + 1], base)
        qrs.append((off[k] - on[k]) / fs * 1000)
        qt.append((te - on[k]) / fs * 1000)
        if sinus and t_prev is not None and np.isfinite(t_prev):
            pr.append((on[k] - p_onset(x20, fs, on[k], t_prev + 0.02 * fs)) / fs * 1000)
        t_prev = te
    rr_s = np.mean(np.diff(r)) / fs
    out = {"hr_bpm": 60 / rr_s, "rr_s": rr_s,
           "pr_ms": np.nanmedian(pr) if np.isfinite(pr).any() else np.nan,
           "qrs_ms": np.median(qrs), "qt_ms": np.nanmedian(qt)}
    out["qtc_bazett_ms"] = out["qt_ms"] / np.sqrt(rr_s)
    out["qtc_fridericia_ms"] = out["qt_ms"] / np.cbrt(rr_s)
    return out


def compare(ecg):
    m = measure_ecg(ecg.leads.to_numpy(), ecg.fs, sinus=ecg.truth["rhythm"] == "sinus")
    keys = ["hr_bpm", "pr_ms", "qrs_ms", "qt_ms", "qtc_bazett_ms", "qtc_fridericia_ms"]
    return pd.DataFrame({"measured": [m[k] for k in keys],
                         "true": [ecg.truth[k] for k in keys]},
                        index=keys).round(1)


compare(case)

**Observe and reflect.** For the case of the week, Bazett's formula gives a QTc above the threshold of 460 ms for women. Fridericia's formula gives a value well below it. At 85 bpm, which correction would you report, and why? The lecture section on QT correction gives the recommendation of the 2009 statement.

In [ ]:
# The five patients of the interactive lab, Part C
LAB_PATIENTS = ["VC-0317", "VC-0079", "VC-0254", "VC-0188", "VC-0018"]
rows = []
for pid in LAB_PATIENTS:
    e = vclinic.patient_ecg(clinic, pid)
    m = measure_ecg(e.leads.to_numpy(), e.fs, sinus=e.truth["rhythm"] == "sinus")
    rows.append({"patient": pid, "conduction": e.truth["conduction"],
                 "HR": e.truth["hr_bpm"], "PR_true": e.truth["pr_ms"], "PR_alg": m["pr_ms"],
                 "QRS_true": e.truth["qrs_ms"], "QRS_alg": m["qrs_ms"],
                 "QT_true": e.truth["qt_ms"], "QT_alg": m["qt_ms"]})
lab_table = pd.DataFrame(rows).set_index("patient").round(1)
lab_table

The last patient, VC-0018, has a left bundle branch block (LBBB). Its QRS is wide, and its T wave in lead II is small and opposite to the main QRS direction. The tangent method in lead II is therefore unreliable for this patient, and the advanced path below shows why.

In [ ]:
# Validation on 100 patients of the clinic (sinus rhythm, normal conduction)
sample = clinic[(clinic["ecg_rhythm"] == "sinus")
                & (clinic["ecg_conduction"] == "normal")].head(100)
errors, qt_pairs = [], []
for pid in sample["patient_id"]:
    e = vclinic.patient_ecg(clinic, pid)
    m = measure_ecg(e.leads.to_numpy(), e.fs)
    errors.append({k: m[k] - e.truth[k] for k in ["pr_ms", "qrs_ms", "qt_ms"]})
    qt_pairs.append((m["qt_ms"], e.truth["qt_ms"]))
errors = pd.DataFrame(errors)
qt_pairs = np.array(qt_pairs)
summary = pd.DataFrame({"bias (ms)": errors.mean(),
                        "SD (ms)": errors.std(),
                        "largest ": errors.abs().max()}).round(1)
summary

**Your calipers against the algorithm and the truth.** The tab *Reflect and export* of the interactive lab exports your caliper measurements of Part C as `my_caliper_measurements.csv`. Save the file in the folder of this notebook and run the next cell. Where a download is not possible, the button *Copy caliper measurements* copies the same text: Paste it between the triple quotes of `PASTED_CALIPERS` in the next cell. Without your own measurements, the cell uses an example file of simulated manual measurements.

In [ ]:
# Bland-Altman analysis of the QT interval
import io
import os
from vclinic.datasets import data_path

# paste the copied caliper measurements between the triple quotes
PASTED_CALIPERS = """
"""

own = "my_caliper_measurements.csv"
if PASTED_CALIPERS.strip():
    source = "PASTED_CALIPERS"
    calipers = pd.read_csv(io.StringIO(PASTED_CALIPERS.strip()))
elif os.path.exists(own):
    source = own
    calipers = pd.read_csv(own)
else:
    source = data_path("examples", "example_caliper_measurements.csv")
    calipers = pd.read_csv(source)
print("Measurements from:", source)
qt_manual = calipers[calipers["measure"] == "QT"].set_index("patient_id")["value_ms"]
pts = [p for p in qt_manual.index if p != "VC-0018"]
truth_qt = [vclinic.patient_ecg(clinic, p).truth["qt_ms"] for p in pts]
alg_qt = [lab_table.loc[p, "QT_alg"] for p in pts]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
teach.plot_bland_altman(axes[0], qt_manual[pts].to_numpy(), truth_qt)
axes[0].set_title("Calipers minus truth")
teach.plot_bland_altman(axes[1], qt_pairs[:, 0], qt_pairs[:, 1])
axes[1].set_title("Algorithm minus truth, 100 patients")
plt.tight_layout()
plt.show()

**Python note.** `elif` means *else if*: Python tests the conditions from top to bottom and runs only the first block whose condition is true. `io.StringIO` makes a piece of text behave like a file, so `pd.read_csv` can read pasted text in the same way as a file on disk.

**Exercise F.** Patient VC-0254 has a QT of 343 ms at a heart rate of 98 bpm in the registry. Compute its QTc with Fridericia's formula from these two numbers and round to one decimal.

In [ ]:
# The table "data" for the prompt of this exercise (Part F of the interactive lab)
data = pd.DataFrame({"patient": ["VC-0254"], "heart rate": [98], "QT": [343]})
answer = None   # your answer
teach.check("Exercise F", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(343 / np.cbrt(60 / 98), 1)
teach.check("Exercise F", answer)

**Exercise G.** Measure the PR interval of patient VC-0188 with `measure_ecg` and round it to the nearest millisecond. Does the result meet the definition of a first-degree atrioventricular block?

In [ ]:
# The table "data" for the prompt of this exercise (Part F of the interactive lab)
ecg_g = vclinic.patient_ecg(clinic, "VC-0188")
m_g = measure_ecg(ecg_g.leads.to_numpy(), ecg_g.fs)
data = pd.DataFrame({"heart rate": [round(m_g["hr_bpm"], 1)], "PR": [round(m_g["pr_ms"], 1)],
                     "QRS": [round(m_g["qrs_ms"], 1)], "QT": [round(m_g["qt_ms"], 1)]})
answer = None   # your answer
teach.check("Exercise G", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
e = vclinic.patient_ecg(clinic, "VC-0188")
answer = round(measure_ecg(e.leads.to_numpy(), e.fs)["pr_ms"])
teach.check("Exercise G", answer)
print("First-degree block (PR > 200 ms):", answer > 200)

**Advanced (optional).** The 2009 recommendations advise measuring the QT in the lead with the longest QT, which is often V2 or V3. Extend `measure_ecg` so that it measures the QT in leads II, V2, V3 and V5 and reports the longest. Try it on VC-0018 and look at the T waves of each lead: Why does the tangent method struggle when the T wave is opposite to the QRS?

In [ ]:
# Advanced (optional): the QT in four leads
e = vclinic.patient_ecg(clinic, "VC-0018")
for lead in ["II", "V2", "V3", "V5"]:
    print(lead, round(measure_ecg(e.leads.to_numpy(), e.fs, lead=lead)["qt_ms"], 1))
print("true QT:", e.truth["qt_ms"])

**The Python path of Practice stop 2 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#real-data-physionet-records) and continue reading at «Access to the PhysioNet and MIT-BIH data sets».

## 6. Access to PhysioNet and MIT-BIH, reading and processing with wfdb, and R-peak detection with NeuroKit2: Record 100

*Lecture: [Access to the PhysioNet and MIT-BIH data sets](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#real-data-physionet-records) · [Reading and processing ECGs with the wfdb library](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#reading-and-processing-ecgs-with-the-wfdb-library) · [R-peak detection (NeuroKit2)](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#finding-every-beat-r-peak-detection)*

**From the formulas to the code.** The width of the integration window, $N = 0.150 \cdot f_s$, is `int(round(0.150 * fs))`, which is 54 samples at 360 Hz. Sensitivity $Se = TP / (TP + FN)$ and positive predictive value $PPV = TP / (TP + FP)$ are computed from the counts that `vclinic.measure.match_beats` returns. A detected beat counts as true when it lies within 150 ms of a reference beat.

Record 100 of the Massachusetts Institute of Technology and Beth Israel Hospital (MIT-BIH) Arrhythmia Database lasts 30 minutes and holds two channels at 360 Hz. Cardiologists annotated every beat. The record is bundled with the course, so the section runs without an internet connection.

**Started here?** Run the catch-up cell below once. It runs the code that Section 6 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB01_python_and_ecg_basics.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(6)

In [ ]:
from vclinic.datasets import read_physionet, read_physionet_annotation
from vclinic.measure import match_beats

mit, fs_mit, notes = read_physionet("100", "mitdb")
ann_samples, ann_symbols = read_physionet_annotation("100", "mitdb")
print(mit.shape, fs_mit, notes)
print(pd.Series(ann_symbols).value_counts())
mit.head()

The annotation symbols follow the convention of PhysioNet: `N` is a normal beat, `A` an atrial premature beat and `V` a premature ventricular contraction. `+` is not a beat; it marks a change of rhythm. The `wfdb` package reads the record when it is installed; otherwise the small reader of `vclinic.physionet` does the same work.

**Reading with wfdb.** `wfdb.rdrecord` reads the header and the signal file of a record, and `wfdb.rdann` reads its annotation file. The record keeps the sampling frequency in `fs`, the names of the signals in `sig_name`, their units in `units` and the signals in millivolts in `p_signal`. With `sampto`, only the first samples are read, here the first ten seconds, and `wfdb.plot_wfdb` draws them with the symbols of the annotations.

In [ ]:
# Reading record 100 directly with wfdb
try:
    import wfdb
    from vclinic.datasets import data_path

    path = str(data_path("physionet", "mitdb", "100"))
    record = wfdb.rdrecord(path)
    annotation = wfdb.rdann(path, "atr")
    print("fs:", record.fs, "| signals:", record.sig_name, "| units:", record.units)
    print("samples per signal:", record.sig_len, "| header comments:", record.comments)
    print("labels:", len(annotation.sample), "| first symbols:", annotation.symbol[:8])
    first10 = wfdb.rdrecord(path, sampto=int(10 * record.fs))
    ann10 = wfdb.rdann(path, "atr", sampto=int(10 * record.fs))
    fig = wfdb.plot_wfdb(record=first10, annotation=ann10, plot_sym=True, time_units="seconds",
                         title="MIT-BIH 100", figsize=(11, 5), return_fig=True)
    for ax, name in zip(fig.axes, first10.sig_name):
        ax.set_ylabel(name + " (mV)")
    fig.axes[-1].set_xlabel("Time (s)")
    plt.show()
except ImportError:
    print("wfdb is not installed; the reader of vclinic has read the same record above.")

**Processing with wfdb.** `processing.resample_singlechan` resamples one signal to a new frequency and moves the annotations to the new samples. `processing.xqrs_detect` finds the QRS complexes with XQRS, the detector of the package.

In [ ]:
# Processing with wfdb: resampling to 250 Hz and the XQRS detector
try:
    from wfdb import processing

    x360 = record.p_signal[:, 0]
    x250, ann250 = processing.resample_singlechan(x360, annotation, record.fs, 250)
    print("samples at 360 Hz:", len(x360), "-> at 250 Hz:", len(x250))
    print("first label at sample:", annotation.sample[0], "->", ann250.sample[0])
    xqrs_peaks = processing.xqrs_detect(x360, fs=record.fs, verbose=False)
    print("XQRS:", len(xqrs_peaks), "QRS complexes")
except (ImportError, NameError):
    xqrs_peaks = None
    print("wfdb is not installed; this step is skipped.")

In [ ]:
# The four stages of the detector on the first 3 seconds
x = mit["MLII"].to_numpy()
seg = slice(0, int(3 * fs_mit))
y = bandpass(x, fs_mit, 5, 15)
z = (np.gradient(y) * fs_mit) ** 2
w = int(round(0.150 * fs_mit))
m = np.convolve(z, np.ones(w) / w, mode="same")
t = np.arange(len(x)) / fs_mit
fig, axes = plt.subplots(4, 1, figsize=(11, 7), sharex=True)
names = ["raw lead MLII (mV)", "band-pass 5-15 Hz",
         "squared slope", f"moving window ({w} samples)"]
for ax, sig, name in zip(axes, [x, y, z, m], names):
    ax.plot(t[seg], sig[seg], color="#9B1B1B" if ax is axes[3] else "k", lw=0.9)
    ax.set_ylabel(name, fontsize=8)
axes[3].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()

In [ ]:
# Detection on the whole recording and comparison with the annotations
x = mit["MLII"].to_numpy()
beat_symbols = set("NLRBAaJSVrFejnE/fQ?")
ref = np.array([s for s, sym in zip(ann_samples, ann_symbols) if sym in beat_symbols])
found = detect_r_peaks(x, fs_mit)
tp, fp, fn = match_beats(found / fs_mit, ref / fs_mit, window_s=0.150)
print(f"reference beats: {len(ref)}, detected: {len(found)}")
print(f"TP = {tp}, FP = {fp}, FN = {fn}")
print(f"Se = {100 * tp / (tp + fn):.2f} %, PPV = {100 * tp / (tp + fp):.2f} %")

**R-peak detection with NeuroKit2.** `nk.ecg_clean` cleans the signal, by default with a high-pass filter at 0.5 Hz and a filter for the power line, and `nk.ecg_peaks` finds the R peaks with its default method. The table compares three detectors on the same annotated beats: the detector of the course, XQRS of wfdb and NeuroKit2.

In [ ]:
# R-peak detection with NeuroKit2, and three detectors compared
rows = [("detector of the course", found)]
if globals().get("xqrs_peaks") is not None:
    rows.append(("XQRS (wfdb)", np.asarray(xqrs_peaks)))
try:
    import neurokit2 as nk

    cleaned = nk.ecg_clean(x, sampling_rate=fs_mit)
    _, info = nk.ecg_peaks(cleaned, sampling_rate=fs_mit)
    nk_peaks = np.asarray(info["ECG_R_Peaks"])
    rows.append(("NeuroKit2", nk_peaks))
except ImportError:
    nk_peaks = None
    print("neurokit2 is not installed; pip install neurokit2 installs it.")
table = []
for name, peaks in rows:
    tp, fp, fn = match_beats(peaks / fs_mit, ref / fs_mit, window_s=0.150)
    table.append({"detector": name, "detected": len(peaks), "TP": tp, "FP": fp, "FN": fn,
                  "Se (%)": round(100 * tp / (tp + fn), 2), "PPV (%)": round(100 * tp / (tp + fp), 2)})
pd.DataFrame(table)

**Exercise H.** How many atrial premature beats (symbol `A`) does record 100 hold?

In [ ]:
# The table "data" for the prompt of this exercise (Part F of the interactive lab)
data = pd.DataFrame({"time": np.asarray(ann_samples) / fs_mit, "label": ann_symbols})
answer = None   # your answer
teach.check("Exercise H", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = ann_symbols.count("A")
teach.check("Exercise H", answer)

**Observe and reflect.** The detector finds every beat of record 100. Why is this result not enough to trust it on every recording? Think of noise, of wide ventricular beats and of tall T waves. Which records of a test set would you choose to challenge it?

**Advanced (optional).** With an internet connection, `wfdb.rdrecord("105", pn_dir="mitdb")` and `wfdb.rdann("105", "atr", pn_dir="mitdb")` read record 105, which is known for its noise, straight from PhysioNet. Run the detector on it and compare the result with `neurokit2.ecg_peaks` if neurokit2 is installed.

In [ ]:
# Advanced (optional): record 105 from PhysioNet (needs internet and wfdb)
try:
    import wfdb
    rec = wfdb.rdrecord("105", pn_dir="mitdb", sampto=360 * 300)
    ann = wfdb.rdann("105", "atr", pn_dir="mitdb", sampto=360 * 300)
    ref105 = np.array([s for s, sym in zip(ann.sample, ann.symbol) if sym in beat_symbols])
    found105 = detect_r_peaks(rec.p_signal[:, 0], rec.fs)
    print(match_beats(found105 / rec.fs, ref105 / rec.fs))
except Exception as err:
    print("Not available here:", type(err).__name__)

## 7. Heart rate variability (HRV) calculation

*Lecture: [Heart rate variability (HRV) calculation](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#heart-rate-variability)*

**From the formulas to the code.** With the RR intervals in milliseconds in an array `rr_ms`, the standard deviation of the normal-to-normal intervals, $SDNN = \sqrt{\frac{1}{N-1}\sum (RR_i - \overline{RR})^2}$, is `np.std(rr_ms, ddof=1)`. The root mean square of successive differences, $RMSSD = \sqrt{\frac{1}{N-1}\sum (RR_{i+1} - RR_i)^2}$, is `np.sqrt(np.mean(np.diff(rr_ms) ** 2))`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 7 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB01_python_and_ecg_basics.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(7)

In [ ]:
# The worked example of the lecture: six RR intervals (ms)
rr_ms = np.array([812, 845, 830, 798, 860, 841])
print("mean RR:", rr_ms.mean())
print("SDNN:", round(np.std(rr_ms, ddof=1), 2), "ms")
print("successive differences:", np.diff(rr_ms))
print("RMSSD:", round(np.sqrt(np.mean(np.diff(rr_ms) ** 2)), 2), "ms")

In [ ]:
# Five minutes of lead II for the case, as the 1996 Task Force recommends for short-term HRV
from vclinic.ecg import lead_ii_long

t5, x5, truth5 = lead_ii_long(case.params, duration_s=300, fs=250, seed=17)
r5 = detect_r_peaks(x5, 250)
rr5 = np.diff(r5) / 250 * 1000                 # RR intervals in ms
sdnn = np.std(rr5, ddof=1)
rmssd = np.sqrt(np.mean(np.diff(rr5) ** 2))
print(f"beats: {len(r5)}, mean HR: {60000 / rr5.mean():.1f} bpm, SDNN: {sdnn:.1f} ms, RMSSD: {rmssd:.1f} ms")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(r5[1:] / 250, rr5, color="#9B1B1B", lw=0.8)
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("RR (ms)")
axes[0].set_title("Tachogram")
axes[1].scatter(rr5[:-1], rr5[1:], s=6, color="#1F5F8B")
axes[1].set_xlabel("RR$_i$ (ms)")
axes[1].set_ylabel("RR$_{i+1}$ (ms)")
axes[1].set_title("Poincaré plot")
plt.tight_layout()
plt.show()

In [ ]:
# Record 100: normal-to-normal intervals of the first 5 minutes
pairs = [(s1, s2) for (s1, a), (s2, b) in zip(zip(ann_samples, ann_symbols),
                                               zip(ann_samples[1:], ann_symbols[1:]))
         if a == "N" and b == "N" and s2 <= 300 * fs_mit]
nn_ms = np.array([(s2 - s1) / fs_mit * 1000 for s1, s2 in pairs])
print(f"NN intervals: {len(nn_ms)}, mean HR: {60000 / nn_ms.mean():.1f} bpm")
print(f"SDNN: {np.std(nn_ms, ddof=1):.1f} ms, RMSSD: {np.sqrt(np.mean(np.diff(nn_ms) ** 2)):.1f} ms")

**HRV with NeuroKit2.** `nk.hrv_time` computes SDNN, RMSSD and many other measures from R peaks, or from RR intervals in milliseconds with their times. The table compares its result for all detected beats of the first five minutes with its result for the NN intervals alone and with the calculation by hand.

In [ ]:
# HRV with NeuroKit2: all detected beats against the NN intervals
try:
    import neurokit2 as nk

    peaks_nk = globals().get("nk_peaks")
    if peaks_nk is None:
        _, info = nk.ecg_peaks(nk.ecg_clean(mit["MLII"].to_numpy(), sampling_rate=fs_mit), sampling_rate=fs_mit)
        peaks_nk = np.asarray(info["ECG_R_Peaks"])
    all_beats = nk.hrv_time(peaks_nk[peaks_nk <= 300 * fs_mit], sampling_rate=fs_mit)
    nn_times = np.array([s2 / fs_mit for s1, s2 in pairs])
    nn_only = nk.hrv_time({"RRI": nn_ms, "RRI_Time": nn_times})
    summary = pd.DataFrame({
        "SDNN (ms)": [all_beats["HRV_SDNN"].iloc[0], nn_only["HRV_SDNN"].iloc[0], np.std(nn_ms, ddof=1)],
        "RMSSD (ms)": [all_beats["HRV_RMSSD"].iloc[0], nn_only["HRV_RMSSD"].iloc[0], np.sqrt(np.mean(np.diff(nn_ms) ** 2))]},
        index=["NeuroKit2, all detected beats",
               "NeuroKit2, NN intervals",
               "by hand, NN intervals"]).round(1)
    display(summary)
except ImportError:
    print("neurokit2 is not installed; pip install neurokit2 installs it.")

**Observe and reflect.** The RMSSD of all detected beats is about twice the RMSSD of the NN intervals. Which beats of the first five minutes cause this difference, and why does the definition of short-term HRV use NN intervals only?

**Explore.** The generator of the virtual clinic has a variability parameter `hrv`: 1.0 is a typical adult at rest, and smaller values mimic older age or heart failure. Move it and watch SDNN and RMSSD of a 5-minute recording.

In [ ]:
# Explore: variability and the two measures
def show_hrv(hrv=1.0, heart_rate=70):
    params = dict(case.params, hrv=hrv, hr_bpm=heart_rate)
    _, xs, _ = lead_ii_long(params, duration_s=300, fs=250, seed=5)
    rr = np.diff(detect_r_peaks(xs, 250)) / 250 * 1000
    print(f"hrv = {hrv:.2f}, HR = {heart_rate} bpm -> SDNN = {np.std(rr, ddof=1):.1f} ms, "
          f"RMSSD = {np.sqrt(np.mean(np.diff(rr) ** 2)):.1f} ms")


try:
    from ipywidgets import interact, FloatSlider, IntSlider
    interact(show_hrv, hrv=FloatSlider(value=1.0, min=0.2, max=1.6, step=0.1),
             heart_rate=IntSlider(value=70, min=50, max=110, step=5, description="HR"))
except ImportError:
    show_hrv(1.0, 70)
    show_hrv(0.4, 70)

**Exercise I.** Compute the RMSSD of the RR intervals 1000, 960, 1010, 990 and 1030 ms, and round it to two decimals.

In [ ]:
answer = None   # your answer
teach.check("Exercise I", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
rr_ex = np.array([1000, 960, 1010, 990, 1030])
answer = round(float(np.sqrt(np.mean(np.diff(rr_ex) ** 2))), 2)
teach.check("Exercise I", answer)

**Observe and reflect.** RMSSD reacts mainly to fast, beat-to-beat changes, and SDNN also to slow ones. Why does the 1996 Task Force recommend recordings of 5 minutes for short-term measures, and why should values from recordings of different lengths not be compared?

## 8. Visualising the 12 leads: A real 12-lead ECG from the PTB Diagnostic database

*Lecture: [Visualising the 12 leads](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#visualising-the-12-leads)*

Record `s0010_re` of patient 1 of the PTB Diagnostic ECG Database, from the Physikalisch-Technische Bundesanstalt (PTB) in Germany, holds the 12 standard leads and three Frank leads at 1000 Hz for 38.4 seconds. Its header describes the patient. The section draws the 12 leads twice: first one below the other, then in the standard printout of the clinic. The record also shows how the steps of Section 5 behave on a real recording.

**Started here?** Run the catch-up cell below once. It runs the code that Section 8 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB01_python_and_ecg_basics.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(8)

In [ ]:
ptb, fs_ptb, header = read_physionet("s0010_re", "ptbdb/patient001")
print(ptb.shape, fs_ptb)
for line in header[:9]:
    print(line)

In [ ]:
# The twelve leads of the real record (column names are lower case)
names = {"I": "i", "II": "ii", "III": "iii", "aVR": "avr", "aVL": "avl", "aVF": "avf",
         "V1": "v1", "V2": "v2", "V3": "v3", "V4": "v4", "V5": "v5", "V6": "v6"}
fig = teach.plot_12lead(ptb.index.to_numpy(), ptb, title="PTB patient001, s0010_re",
                        v_range=(-1.5, 1.5), names=names)
plt.show()

**The standard printout.** A 12-lead ECG is printed in three rows of four columns: I, II and III in the first column, aVR, aVL and aVF in the second, V1 to V3 in the third and V4 to V6 in the fourth. Each lead fills 2.5 seconds, and a fourth row shows lead II for all 10 seconds as a rhythm strip. On the paper, 25 mm/s and 10 mm/mV make a small square 0.04 s wide and 0.1 mV high, and a large square of five small squares 0.2 s wide and 0.5 mV high.

**Python note.** `plt.subplots(4, 1)` creates four rows of axes. `(t >= 2.5 * col) & (t < 2.5 * (col + 1))` is a Boolean mask: It is true for the samples of one 2.5-second column. `ax.set_aspect(0.4)` keeps the proportion of the paper, because 1 mV is drawn as 10 mm and 1 s as 25 mm, and 10 / 25 = 0.4.

In [ ]:
# The standard printout: 3 × 4 layout with a rhythm strip of lead II at 25 mm/s and 10 mm/mV
layout = [["I", "aVR", "V1", "V4"], ["II", "aVL", "V2", "V5"], ["III", "aVF", "V3", "V6"]]
shown = {"I": "I", "II": "II", "III": "III"}
ten = ptb.iloc[: int(10 * fs_ptb)]                               # the first 10 seconds
t = ten.index.to_numpy()
fig, axes = plt.subplots(4, 1, figsize=(12, 7.6))
for row, leads_row in enumerate(layout):
    for col, lead in enumerate(leads_row):
        part = (t >= 2.5 * col) & (t < 2.5 * (col + 1))         # each lead fills 2.5 s
        axes[row].plot(t[part], ten[names[lead]].to_numpy()[part], color="k", lw=0.8)
        axes[row].text(2.5 * col + 0.06, 1.62, shown.get(lead, lead), fontsize=9)
axes[3].plot(t, ten["ii"].to_numpy(), color="k", lw=0.8)         # rhythm strip
axes[3].text(0.06, 1.62, shown["II"], fontsize=9)
for ax in axes:
    ax.set_xlim(0, 10)
    ax.set_ylim(-1.5, 2.0)
    ax.set_xticks(np.arange(0, 10.001, 0.2))                      # large square: 0.2 s
    ax.set_xticks(np.arange(0, 10.001, 0.04), minor=True)         # small square: 0.04 s
    ax.set_yticks(np.arange(-1.5, 2.001, 0.5))                    # large square: 0.5 mV
    ax.set_yticks(np.arange(-1.5, 2.001, 0.1), minor=True)        # small square: 0.1 mV
    ax.grid(which="major", color="#E8A0A0", lw=0.7)
    ax.grid(which="minor", color="#F6DADA", lw=0.4)
    ax.tick_params(which="both", labelbottom=False, labelleft=False, length=0)
    ax.set_aspect(0.4)                                            # 1 mV = 10 mm and 1 s = 25 mm
axes[3].set_xlabel("10 s at 25 mm/s and 10 mm/mV")
plt.tight_layout()
plt.show()

In [ ]:
# Einthoven's and Goldberger's relations on real data
print("max |II - (I + III)| =", round(float(np.abs(ptb["ii"] - (ptb["i"] + ptb["iii"])).max()), 4), "mV")
print("max |aVR + aVL + aVF| =", round(float(np.abs(ptb["avr"] + ptb["avl"] + ptb["avf"]).max()), 4), "mV")

In [ ]:
# The measurement of Section 5 on the real record
leads_ptb = ptb[[names[k] for k in vclinic.LEADS]].to_numpy()
m_ptb = measure_ecg(leads_ptb, fs_ptb)
pd.Series(m_ptb).round(1)

**Observe and reflect.** The algorithm measures a QRS duration of about 137 ms in this record. Look at lead V1 and at the inferior leads in the figure. Which pattern could explain a QRS of this width, and what do the inferior leads show? The 2009 recommendations ask that computer measurements be checked by eye: Measure one beat yourself on the figure and compare.

**Advanced (optional).** The three Frank leads `vx`, `vy` and `vz` record the heart vector almost directly. Plot the QRS loop of one beat in three dimensions. Textbooks use different sign conventions for the Frank leads, so the loop is drawn here as recorded.

In [ ]:
# Advanced (optional): the QRS loop of one beat from the Frank leads
r_ptb = detect_r_peaks(ptb["ii"].to_numpy(), fs_ptb)
k = r_ptb[3]
loop = ptb[["vx", "vy", "vz"]].to_numpy()[k - int(0.06 * fs_ptb):k + int(0.08 * fs_ptb)]
fig = plt.figure(figsize=(5.5, 5))
ax = fig.add_subplot(projection="3d")
ax.plot(loop[:, 0], loop[:, 1], loop[:, 2], color="#9B1B1B")
ax.set_xlabel("vx (mV)")
ax.set_ylabel("vy (mV)")
ax.set_zlabel("vz (mV)")
plt.show()

## 9. Working with an AI assistant: The code safety checklist

*Lecture: [Working with an AI assistant, part 1: The code safety checklist](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#ai-pair-programming-part-1-the-code-safety-checklist)*

An AI assistant can write most of the code of this course, and it can also write code that runs without error and returns a wrong number. The course therefore treats AI-written code like a new member of the team: It is welcome, and it is checked. The prompt card below gives the structure of a good request; it is the prompt of step 3.4 of the prompt path.

**Prompt card.** *Role:* You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there. *Context:* On an ECG sampled 500 times per second, the QT interval and the RR interval were measured as numbers of samples. The cardiologist wants the QT interval corrected by Bazett's method, in milliseconds. *Data:* The notebook holds a table named "data" with one row: The column "QT" gives the QT interval and the column "RR" the RR interval, both as numbers of samples. The recording was sampled 500 times per second. *Task:* Turn the QT interval into milliseconds and the RR interval into seconds. Correct the QT by Bazett's method: Divide the QT in milliseconds by the square root of the RR interval in seconds. *Constraints:* Use only this table. Do not assume the number of samples per second; use the 500 given here. The code must not read files, download anything from the internet or install programs. *Output:* Print the corrected QT in milliseconds with one decimal and keep it under the name "answer". *Test:* The same calculation must give exactly 400 ms for a QT of 200 samples and an RR interval of 500 samples, because the RR interval is then exactly 1 second.

**Check before using.** The lecture page gives the code safety checklist of the course: *sign in* before running (units, sampling frequency, lead, no patient identifier), *time out* while running (errors, physiological values) and *sign out* before using (comparison with a known truth, every line explained, prompt and changes logged).

An AI assistant produced the function below from a shorter prompt that gave no sampling frequency.

**Started here?** Run the catch-up cell below once. It runs the code that Section 9 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB01_python_and_ecg_basics.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(9)

In [ ]:
# Code written by an AI assistant from a short prompt
def qtc_from_samples(qt_samples, rr_samples, fs=360):
    qt_ms = qt_samples / fs * 1000
    rr_ms = rr_samples / fs * 1000
    return qt_ms / np.sqrt(rr_ms)


print(qtc_from_samples(196, 353))

**Observe and reflect.** The function runs without an error. Go through the checklist: Which pause catches the result, and what are the two errors? One of them comes from a default value that the prompt never asked for.

**Exercise J.** Correct the function: Take `fs` without a default value and use the RR interval in seconds. Compute the QTc for 196 and 353 samples at 500 Hz and round to one decimal.

In [ ]:
# The table "data" for the prompt of this exercise (Part F of the interactive lab)
data = pd.DataFrame({"QT": [196], "RR": [353]})
def qtc_from_samples_fixed(qt_samples, rr_samples, fs):
    return None   # your corrected formula


answer = qtc_from_samples_fixed(196, 353, 500)
answer = round(answer, 1) if answer is not None else None
teach.check("Exercise J", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def qtc_from_samples_fixed(qt_samples, rr_samples, fs):
    """Bazett QTc (ms) from QT and RR given in samples at fs (Hz)."""
    qt_ms = qt_samples / fs * 1000
    rr_s = rr_samples / fs
    return qt_ms / np.sqrt(rr_s)


answer = round(qtc_from_samples_fixed(196, 353, 500), 1)
teach.check("Exercise J", answer)

**AI usage log.** Each weekly project adds a short log with one row per use of an AI assistant:

| Tool | Prompt (short) | What was checked | What was changed |
|---|---|---|---|
| (name of the assistant) | Bazett QTc from samples | Result 466.5 ms against the true values of VC-0317 | Removed the default fs; RR in seconds |

**The Python path of Practice stop 3 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#the-paper-of-the-week) and continue reading at «The paper of the week». The rest of this Jupyter notebook holds further practice and the help for the weekly project.

## 10. Application key: Another patient, another lead

Change `PATIENT` and `LEAD` and run the cell again. Patients worth a look: VC-0183 (a long QT interval), VC-0018 (left bundle branch block), VC-0207 (a PR interval of 218 ms in the registry) and any patient whose `ecg_rhythm` is `AF`. `clinic[clinic["stemi_territory"] != "none"]` lists the patients with an acute ST-elevation myocardial infarction (STEMI), the topic of Week 5.

**Started here?** Run the catch-up cell below once. It runs the code that Section 10 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB01_python_and_ecg_basics.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(10)

In [ ]:
PATIENT = "VC-0183"
LEAD = "II"

e = vclinic.patient_ecg(clinic, PATIENT)
row = clinic[clinic["patient_id"] == PATIENT].iloc[0]
print(PATIENT, row["age"], row["sex"], row["main_diagnosis"], row["ecg_rhythm"], row["ecg_conduction"])
m = measure_ecg(e.leads.to_numpy(), e.fs, lead=LEAD, sinus=e.truth["rhythm"] == "sinus")
keys = ["hr_bpm", "pr_ms", "qrs_ms", "qt_ms", "qtc_bazett_ms", "qtc_fridericia_ms"]
display(pd.DataFrame({"measured": [m[k] for k in keys],
                      "true": [e.truth[k] for k in keys]}, index=keys).round(1))
fig = teach.plot_12lead(e.time, e.leads, title=PATIENT)
plt.show()

## Application challenges by field

| Field | Challenge |
|---|---|
| Electrophysiology | Generate the ECG of VC-0183 at heart rates from 50 to 110 bpm (change `hr_bpm` in `e.params` and keep `qt_ms` scaled by $RR^{1/3}$). Plot QTc by Bazett and by Fridericia against heart rate and decide which one to report in a QT monitoring protocol. |
| Interventional cardiology | List the patients with a STEMI territory, draw their 12-lead ECGs and name the leads with ST elevation for each territory. Compare with the culprit artery that the territory suggests. |
| Heart failure | Compare SDNN and RMSSD of 5-minute recordings between the HFrEF patients and the patients without heart disease of the clinic. |
| Cardio-oncology | Build a QTc follow-up table for one patient on a QT-prolonging treatment, with serial ECGs at different heart rates, and flag the visits at which the QTc crosses the threshold of your local protocol. |
| Pediatric cardiology | The 2009 recommendations use lower QRS thresholds in children. Change the threshold of a wide QRS in a function of your own and discuss what else would change in a child's ECG. |
| Sports cardiology | Generate a resting ECG with a heart rate of 45 bpm and compare QTc by both formulas: Which one could label an athlete as abnormal? |
| Intensive care | Generate 5 minutes with `noise="clinic"` and measure Se and PPV of the R-peak detector against the true beat times. |

## After the notebook

Open the [interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lab.html) and take the self-assessment in the tab *Check yourself*. Then write the reflection in the tab *Reflect and export* and export the learning log. The weekly project of Week 1, described on the [week page](README.md), starts from the function `measure_ecg` of Section 5.

<div align="center">

---

**Prof. Dr. Utku Kose**<br>
Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University<br>
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists, Turkish Society of Cardiology. First version: November 2026. Content CC BY 4.0, code MIT.*

</div>